# GAAL Ablation 131J — Ablation Studies + Extended Visualization

Successor to `GAAL_Ablation_130J.ipynb`. **130J is preserved unchanged**; this notebook re-runs the ablation with the proposed variant aligned to the case-study implementation, and writes to a new experiment id so no 126J artifact is overwritten.

## What changed from 130J — and why

130J's `ab8c` was **not the same algorithm** as the case-study implementation (`casestudy/methods/m10_ab8c.py`), although the paper reported them as one method. Three differences:

| | 130J `ab8c` | case-study `m10_ab8c` |
|---|---|---|
| Acquisition | BADGE (k-means++ on gradient embeddings — acquires the *most uncertain* candidates) | confidence-gated diverse (rejects candidates the classifier will not confidently assign to the intended class, then diversifies) |
| Generator | refined once before the AL loop, then frozen | updated per cycle inside the AL loop |
| Candidate seeding | class-conditional prior | latent-SMOTE interpolation of encoded real minority latents |

131J aligns all three. `_align_check_ab8c.py` verifies **25/25 comparable knobs match**.

The reason the acquisition matters: BADGE assumes an oracle relabels each acquired point, so ambiguity is informative. Here there is no oracle — an acquired sample enters training carrying the class it was *generated* for, so an ambiguous generation is injected as label noise.

- **Experiment ID:** `131J`  ·  **Classes:** `[0, 2, 3]` (T-shirt/top, Pullover, Dress)  ·  **Imbalance:** `800 : 40 : 40`
- **Proposed method:** `ab8c` (confidence-gated diverse acquisition + latent-SMOTE seeding + D/G/C co-training on BAGAN-GP)

## Paper Table 4 (four variants)

| Paper name | Mode | Backbone | Component added |
|---|---|---|---|
| Baseline | `ab1` | — | Imbalanced training, no augmentation |
| Variant 1 | `ab8a` | BAGAN | + GAN + AL, aux-head label gate then **margin** selection (class-conditional prior seeding) |
| Variant 2 | `ab8d` | BAGAN | + AL with **confidence-gated diverse** selection (τ = 0.6) + **latent-SMOTE** candidate seeding |
| **Proposed** | `ab8c` | BAGAN-GP | + BAGAN-GP backbone, D/G/C co-training (shares Variant 2's confidence-gated diverse selection + latent-SMOTE seeding) |

> **132J:** Variant 2 (`ab8d`) now uses the same **latent-SMOTE** candidate seeding as Proposed (`ab8c`) — it previously sampled the class-conditional prior like `ab8a`/`ab8b`. This makes the **Variant 2 → Proposed** step a clean, single-factor comparison (BAGAN-GP backbone + D/G/C co-training only). It also means the **Variant 1 → Variant 2** step now changes *two* things — the acquisition rule AND the candidate seeding — not the selector alone. The knob is `config.ab8d_seed_mode` (independent of `ab8c_seed_mode`; `ab8b`, the excluded BADGE mode, is unaffected and keeps sampling the prior).

> **`ab8b` (BADGE) has been removed from the study.** BADGE is an alternative acquisition rule, not a component of the proposed pipeline, so it is not part of the component chain Table 4 presents. Rather than compute it and then decline to report it, it is excluded from the design altogether. Its earlier single-seed numbers remain in `saved/_backup_ablation_rows_131J_*.csv`.

## Run protocol

- **Mode menus:**
  - `GAN_BACKEND='bagan'` → `['ab1', 'ab8a', 'ab8d']`
  - `GAN_BACKEND='bagan_gp'` → `['ab8c']`
- **Run TWICE** (restart kernel between passes): pass 1 `GAN_BACKEND='bagan'`, then pass 2 `GAN_BACKEND='bagan_gp'`. Each pass saves per-backend CSVs + a sample dump; the final cells merge both passes.
- **Metrics:** Recall / G-mean / F2 / `macro_specificity`. **FID:** per-class, plus per-class **autoencoder FID** (one triple per backend pass).

> **Watch the run log for `[selection] confidence gate starved`.** If the gate rejects nearly every candidate, the rule falls back to top-confidence picks and the k-means++ diversity step stops doing useful work — the method would still run, but it would not be the method described.

> **Table 4 is not monotone on F2**, and §I prints a monotonicity check on every run. Do not write "performance improves with the addition of each component" without looking at that output first.


In [ ]:
# Enable automatic module reloading
%load_ext autoreload
%autoreload 2

from datetime import datetime
print(f"\nStart time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

import time   # local alias to avoid shadowing if already imported

time_stamp_1 = time.time()

## §A · INIT — Environment & Data Setup

Import libraries, fix reproducibility (`SEED=42`), choose `GAN_BACKEND`, create `AblationConfig(date_suffix="131J")`, load Fashion-MNIST `[0,2,3]`, remap labels to contiguous `{0,1,2}`, resize to 64×64, normalize to `[-1,1]`, and build the test `DataLoader`.

> **The suffix is `131J`, not `126J`.** 126J owns 130J's saved CSVs and checkpoints; reusing it would overwrite the very results this run is meant to be compared against.

> **Run-twice protocol:** set `GAN_BACKEND='bagan'` for pass 1, restart the kernel, then set `GAN_BACKEND='bagan_gp'` for pass 2. The cell below auto-selects `MODES_TO_RUN` from the active backend (set `SELECTED_MODES` to run a subset).


In [ ]:
# =============================================================================
# §A.1 — Imports, Reproducibility, GAN backend & Configuration
# =============================================================================
# 0. Fix OpenMP duplicate-library crash (MUST precede `import torch`).
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# 1. Standard libraries
import time
from datetime import datetime
import numpy as np
import random
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, TensorDataset

# 2. ablation/ package (backend-agnostic experiment pipeline)
from ablation import (
    AblationConfig,
    create_imbalanced_pool,
    build_test_set,
    dataset_to_tensors,
    print_class_distribution,
    run_kfold_ablation,
)

# 3. ─── GAN BACKEND SWITCH ──────────────────────────────────────────────────
#    Pass 1: 'bagan'     → modes ab1, ab8a, ab8d
#    Pass 2: 'bagan_gp'  → mode  ab8c                  (restart kernel first!)
#
#    Overridable by the multi-seed driver via the GAAL_BACKEND env var, so the
#    same notebook file serves both interactive use and the batch sweep.
GAN_BACKEND = os.environ.get('GAAL_BACKEND', 'bagan')   # 'bagan' or 'bagan_gp'

# 4. Bind BAGAN classes from the chosen package (must happen AFTER the switch).
if GAN_BACKEND == 'bagan':
    from bagan import (
        BAGANConfig, BAGANAutoencoder, BAGANGenerator, BAGANDiscriminator,
        AutoencoderTrainer, GANTrainer, ClassConditionalLatentGenerator, WeightTransfer,
    )
elif GAN_BACKEND == 'bagan_gp':
    from bagan_gp import (
        BAGANConfig, BAGANAutoencoder, BAGANGenerator, BAGANDiscriminator,
        AutoencoderTrainer, GANTrainer, ClassConditionalLatentGenerator, WeightTransfer,
    )
else:
    raise ValueError(f"Unknown GAN_BACKEND={GAN_BACKEND!r}; expected 'bagan' or 'bagan_gp'.")

# 5. ─── Reproducibility: TWO seeds with different jobs ──────────────────────
#    DATA_SEED (always 42) — imbalanced pool, test set, k-fold assignment and
#      the retrain train/val split. HELD FIXED across every seed of the
#      multi-seed study so all seeds and all modes are scored on identical
#      data; that is what makes the variant-to-variant comparison paired.
#    SEED (GAAL_SEED, default 42) — everything stochastic in the *method*:
#      network init, batch order, GAN noise, acquisition RNG, classifier init.
#      This is what VARIES across the study.
#
#    Mirrors the case-study protocol in cs3j_runner.run_one(), where the split
#    is pinned to config.random_seed while the per-run seed drives the method.
#    Difference to disclose in §4.3: the case study additionally pins the
#    classifier to seed 42; here the classifier varies too, so the reported sd
#    also covers classifier init/batch-order noise.
DATA_SEED = 42
SEED = int(os.environ.get('GAAL_SEED', 42))

def set_seed(seed: int = SEED) -> None:
    """Reset ALL random number generators for full reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

set_seed(SEED)
print(f"Data seed (fixed): {DATA_SEED}")
print(f"Run seed (varies): {SEED}")

# 6. AblationConfig — NEW 131J experiment id.
#    Deliberately NOT "126J": that suffix owns 130J's saved CSVs and model
#    checkpoints, and reusing it would overwrite the results this notebook is
#    meant to be compared against.
#
#    DATE_SUFFIX is overridable (GAAL_SUFFIX) so each seed of the sweep writes
#    to its own artifacts, e.g. "131J_s43", instead of clobbering the previous.
# Defaults to the seed-44 five-seed-sweep suffix so a vanilla "Run All"
# (no env vars set) lands on data that already exists on disk and the
# §A.3 auto-detect below can skip straight to reusing it. The multi-seed
# driver (_run_133J_seeds.py) always overrides this via GAAL_SUFFIX, so
# this default has no effect on the sweep.
DATE_SUFFIX = os.environ.get('GAAL_SUFFIX', '133J_s44')

config = AblationConfig(
    date_suffix=DATE_SUFFIX,
    gan_backend=GAN_BACKEND,
    random_seed=DATA_SEED,   # data partitioning — fixed
    run_seed=SEED,           # acquisition RNG   — varies
)
# --- co-training weight diagnostic (133J) -----------------------------------
# The seed-42 run showed the co-trained generator's FID identical to the
# pre-loop generator's, i.e. lambda_anchor (1.0) pins G_theta to G_ref against
# a lambda_cls of only 0.2. These overrides let a run vary that balance without
# editing ablation/config.py; unset, they leave the published defaults intact.
_lam_anchor = os.environ.get('GAAL_LAMBDA_ANCHOR')
_lam_cls    = os.environ.get('GAAL_LAMBDA_CLS')
if _lam_anchor is not None:
    config.ab8c_lambda_anchor = float(_lam_anchor)
if _lam_cls is not None:
    config.ab8c_lambda_cls = float(_lam_cls)
if _lam_anchor is not None or _lam_cls is not None:
    print(f"[co-training override] lambda_anchor={config.ab8c_lambda_anchor} "
          f"lambda_cls={config.ab8c_lambda_cls}")

device = config.get_device()
notebook_start_time = time.time()
print(f"Start time:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"GAN backend:  {GAN_BACKEND}")
print(f"Device:       {device}")
print(f"Date suffix:  {config.date_suffix}")
print(f"Seeds:        random_seed(data)={config.random_seed}  run_seed(method)={config.run_seed}")

# 7. Acquisition rule per Ab-8 mode (131J change; see the header cell).
print("\nAcquisition rules:")
for _m in ('ab8a', 'ab8d', 'ab8c'):
    if _m == 'ab8a':
        print(f"  {_m:<5} aux-head label gate (tau={config.ab8a_aux_tau}) then margin")
        continue
    _p = config.get_selection_params(_m)
    print(f"  {_m:<5} {_p['mode']:<18} "
          f"min_conf={_p['min_confidence']}, normalize={_p['normalize']}, "
          f"pool_factor={_p['pool_factor']}")


In [ ]:
GAN_BACKEND

### §A.2 — Load Data, Remap Labels, Resize & Normalize

Create the imbalanced training pool `800:40:40` and a balanced test set; convert to dense tensors; **remap labels `{0,2,3}` → `{0,1,2}`** (and update `config.selected_classes`); resize 28×28 → 64×64; normalize `[0,1]` → `[-1,1]`.


In [ ]:
# =============================================================================
# §A.2 — Data: imbalanced pool + balanced test, remap, resize, normalize
# =============================================================================
print("=" * 70)
print("  §A.2 — Loading & Preprocessing Data")
print("=" * 70)

# 1. Imbalanced training pool (class 0: 800, classes 2,3: 40 each).
#    ⚠️ Seeded with DATA_SEED, never SEED: the pool and test set must be
#    IDENTICAL across every seed of the multi-seed study, otherwise the
#    variant-to-variant comparison stops being paired and the reported sd
#    conflates method noise with resampling noise.
set_seed(DATA_SEED)
pool_dataset, pool_indices = create_imbalanced_pool(config)
print_class_distribution(pool_dataset, "Training Pool (imbalanced)")

# 2. Test set (same classes; built from the FMNIST test split).
test_dataset = build_test_set(config)
print_class_distribution(test_dataset, "Test Set")

# 3. Dense tensors.
pool_data, pool_labels = dataset_to_tensors(pool_dataset)
test_data, test_labels = dataset_to_tensors(test_dataset)

# 3a. ⚠️ CRITICAL — remap non-contiguous labels {0,2,3} → {0,1,2}.
label_map = {orig: new for new, orig in enumerate(config.selected_classes)}
inverse_label_map = {new: orig for orig, new in label_map.items()}
remapped_class_names = {new: config.class_names[orig] for orig, new in label_map.items()}

pool_labels_original = pool_labels.clone()
test_labels_original = test_labels.clone()
for orig, new in label_map.items():
    pool_labels[pool_labels_original == orig] = new
    test_labels[test_labels_original == orig] = new

# 3b. Update config to the contiguous label space.
config.selected_classes = list(range(config.num_classes))
print(f"\n🔄 Label remap: {label_map}  (inverse {inverse_label_map})")
print(f"   Remapped names: {remapped_class_names}")
print(f"   config.selected_classes = {config.selected_classes}")

# 4. Resize 28×28 → 64×64 and normalize [0,1] → [-1,1] (matches generator Tanh).
pool_data = F.interpolate(pool_data, size=64, mode='bilinear', align_corners=False) * 2 - 1
test_data = F.interpolate(test_data, size=64, mode='bilinear', align_corners=False) * 2 - 1

print(f"\n📐 Pool: data={tuple(pool_data.shape)} range=[{pool_data.min():.2f},{pool_data.max():.2f}]")
print(f"   Test: data={tuple(test_data.shape)} range=[{test_data.min():.2f},{test_data.max():.2f}]")


### §A.3 — Test DataLoader, class labels & backend-driven `MODES_TO_RUN`

Build the held-out test `DataLoader`, define the per-class label/name helpers used by FID, and select which ablation modes this pass runs. Each backend exposes a **full menu**; set `SELECTED_MODES` to a subset (preserving your order) or leave it `None` to run the whole menu.

- `GAN_BACKEND='bagan'` → `['ab1', 'ab8a', 'ab8d']`
- `GAN_BACKEND='bagan_gp'` → `['ab8c']`

`ab8d` (confidence-gated) is Table 4's Variant 2. It shares the vanilla BAGAN backend and generator with `ab8a`, so comparing them isolates the **acquisition rule** — but, since 132J, `ab8d` also switches candidate seeding to **latent-SMOTE** (`ab8a` still samples the class-conditional prior), so the `ab8a` → `ab8d` step now changes selector **and** seeding together. The single-factor comparison is `ab8d` → `ab8c` (Variant 2 → Proposed), which isolates the BAGAN-GP backbone + D/G/C co-training only, since both share the confidence-gated diverse selector and latent-SMOTE seeding.


In [ ]:
# =============================================================================
# §A.3 — Test DataLoader, class helpers, and MODES_TO_RUN (backend-driven)
# =============================================================================
# Held-out test DataLoader (no shuffle — deterministic evaluation order).
test_loader = DataLoader(
    TensorDataset(test_data, test_labels),
    batch_size=config.batch_size,
    shuffle=False,
    pin_memory=(device != 'cpu'),
)

# Per-class helpers (contiguous label space {0,1,2}) used by FID + per-class F1.
CLASS_IDS = list(range(config.num_classes))
CLASS_NAMES = {c: remapped_class_names[c] for c in CLASS_IDS}

# ─── Which modes does THIS backend pass offer? ───────────────────────────────
#   bagan     → ab1 (baseline) + ab8a (margin), ab8d (confidence-gated)
#   bagan_gp  → ab8d (Variant 3: gate + GP backbone, NO co-training)
#               ab8c (conf-gated + latent-SMOTE + D/G/C co-training; proposed)
#
# Table 4 rows (133J, five variants -- each differs from the row above in ONE
# respect, so the last arrow isolates co-training):
#   Baseline  = ab1                      (no augmentation)
#   Variant 1 = ab8a  @ bagan            (+ generation + margin acquisition)
#   Variant 2 = ab8d  @ bagan            (+ confidence-gated selection)
#   Baseline a= ab4   @ bagan / bagan_gp (generate once, NO selection)
#   Variant 1a= ab8a  @ bagan_gp         (margin  + GP backbone)
#   Variant 3 = ab8d  @ bagan_gp         (gate    + GP backbone)
#   Proposed  = ab8c  @ bagan_gp         (+ three-player co-training)
#
# Variants 1/1a/2/3 form a 2x2 over {margin, confidence-gated} x {BAGAN,
# BAGAN-GP}, so the selection-rule effect and the backbone effect are each
# measured twice; Proposed adds co-training on top of Variant 3.
#
# ab8b (BADGE) was REMOVED from the study design. It is not a component of the
# proposed pipeline, so including it would have meant computing a result that
# the paper does not report. Its earlier single-seed numbers are preserved in
# saved/_backup_ablation_rows_131J_*.csv for the record.
MODES_BY_BACKEND = {
    'bagan':    ['ab1', 'ab4', 'ab8a', 'ab8d'],
    'bagan_gp': ['ab4', 'ab8a', 'ab8d', 'ab8c'],
}

# Optional subset override — must be a subset of the active backend's menu.
# Set to None to run the FULL menu for this backend; order is preserved.
SELECTED_MODES = None   # e.g. ['ab1', 'ab8d']
# GAAL_MODES lets a driver restrict the menu without editing this cell,
# e.g. GAAL_MODES='ab4' to run one mode alone for diagnostics.
_env_modes = os.environ.get('GAAL_MODES')
if _env_modes:
    SELECTED_MODES = [m.strip() for m in _env_modes.split(',') if m.strip()]
    print(f'[GAAL_MODES override] {SELECTED_MODES}')

_full_menu = MODES_BY_BACKEND[GAN_BACKEND]
if SELECTED_MODES is None:
    MODES_TO_RUN = list(_full_menu)
else:
    _bad = [m for m in SELECTED_MODES if m not in _full_menu]
    assert not _bad, f"{_bad} invalid for backend {GAN_BACKEND!r}; choose from {_full_menu}"
    MODES_TO_RUN = list(SELECTED_MODES)   # preserve the user's selection order

# ─── Reuse-existing-data auto-detection ──────────────────────────────────
# If saved/ablation_rows_<suffix>_<backend>.csv already exists, §B-§H below
# are redundant: the pipeline is fully seeded, so retraining reproduces the
# same numbers at real GPU cost (verified for suffix=133J_s44: identical to
# 6 decimals across two independent runs). REUSE_EXISTING_DATA auto-detects
# that case so a plain "Run All" is fast and safe by default.
#   GAAL_REUSE_EXISTING=1  -> force reuse/skip regardless of what exists
#   GAAL_REUSE_EXISTING=0  -> force real training regardless of what exists
#                             (this is what the multi-seed driver sets)
#   unset                  -> auto: reuse iff the rows CSV is already there
from pathlib import Path as _Path_reuse
_rows_csv_path = _Path_reuse(config.bagan_model_dir) / f"ablation_rows_{config.date_suffix}_{GAN_BACKEND}.csv"
_reuse_env = os.environ.get('GAAL_REUSE_EXISTING')
if _reuse_env is not None:
    REUSE_EXISTING_DATA = _reuse_env not in ('0', 'false', 'False')
else:
    REUSE_EXISTING_DATA = _rows_csv_path.is_file()
print(f"\nREUSE_EXISTING_DATA: {REUSE_EXISTING_DATA}  "
      f"({'found' if _rows_csv_path.is_file() else 'not found'}: {_rows_csv_path})")
if REUSE_EXISTING_DATA:
    print("  -> §B-§H will SKIP training and reuse the existing rows CSV "
          f"for backend={GAN_BACKEND}.")
else:
    print(f"  -> §B-§H will run the FULL training pipeline for backend={GAN_BACKEND}.")

os.makedirs(config.bagan_model_dir, exist_ok=True)
print(f"GAN backend:   {GAN_BACKEND}")
print(f"Full menu:     {_full_menu}")
print(f"Modes to run:  {MODES_TO_RUN}")
print(f"Class ids:     {CLASS_IDS}")
print(f"Class names:   {CLASS_NAMES}")
print(f"Save dir:      {config.bagan_model_dir}")
print(f"Test batches:  {len(test_loader)}")


---

### ⚠️ This pass: §B–§H intentionally skipped (seed 44, publication-variant refresh)

This execution reuses the already-computed seed-44 data from the five-seed sweep (`saved/ablation_rows_133J_s44_bagan.csv`, `..._bagan_gp.csv`) instead of re-running §B–§H, which would spend roughly 1-2 hours of GPU time to reproduce numbers that are already deterministic and on disk. `GAAL_SUFFIX` was set to `133J_s44` for this run so §I's glob finds those exact files.

§B–§H below therefore show **no output for this pass** (any stale output from an earlier full run has been cleared). §I onward reflects the updated four-variant mapping — Baseline=(bagan,ab1), Variant1=(bagan,ab8a), Variant2=(bagan_gp,ab8d), Proposed=(bagan_gp,ab8c) — against the seed-44 data.

§I's outputs (merged summary + publication CSV) are saved under the `133J_s44_pub4` suffix, not `133J_s44`, so the five-seed sweep's canonical files are never touched by this pass.

## §B · BAGAN config for cross-validation

`run_kfold_ablation` trains a *fresh per-fold BAGAN* for every BAGAN mode (`ab8a`, `ab8d`, `ab8c`) to avoid leakage. It needs a reduced-epoch `bagan_config_cv`. Budget is backend-aware (parity with `124J` §A.4):

| backend | `ae_epochs` | `gan_epochs` |
|---------|-------------|--------------|
| `bagan` | 30 | 200 |
| `bagan_gp` | 50 | 1500 (`300 × n_critic`) |


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §B — BAGAN config for per-fold CV (reduced epochs, backend-aware)
    # =============================================================================
    # BAGANConfig was bound in §A.1 from the chosen backend package.
    # .for_cv() returns a reduced-epoch copy used by run_kfold_ablation()'s
    # per-fold BAGAN training. Non-BAGAN modes (ab1) ignore this.
    if GAN_BACKEND == 'bagan':
        bagan_config_cv = BAGANConfig.from_ablation_config(config).for_cv(ae_epochs=30, gan_epochs=200)
    elif GAN_BACKEND == 'bagan_gp':
        bagan_config_cv = BAGANConfig.from_ablation_config(config).for_cv(ae_epochs=50, gan_epochs=1500)
    else:
        raise ValueError(f"Unknown GAN_BACKEND={GAN_BACKEND!r}")
    bagan_config_cv.save_checkpoints = False  # no disk writes during CV

    print("=" * 70)
    print(f"§B — BAGAN CV config  [backend={GAN_BACKEND}]")
    print("=" * 70)
    print(f"  ae_epochs:   {bagan_config_cv.ae_epochs}")
    print(f"  gan_epochs:  {bagan_config_cv.gan_epochs}")
    print(f"  latent_dim:  {bagan_config_cv.latent_dim}")
    print(f"  device:      {bagan_config_cv.device}")

else:
    print("[reuse] skipped §B \u2014 REUSE_EXISTING_DATA=True")

## §C · Cross-validation (per mode)

Run stratified k-fold CV for every mode in `MODES_TO_RUN`. CV is used **only** to derive each mode's `avg_stopping_epoch` (consumed by the §E retrain). Each mode is reseeded deterministically (`set_seed(SEED)`) so its result is independent of run order. BAGAN modes receive `bagan_config_cv`; `ab1` does not.

In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §C — Cross-validation loop (per mode) → avg_stopping_epoch
    # =============================================================================
    from ablation.kfold import run_kfold_ablation

    # Modes that need a per-fold BAGAN (everything except the raw baseline).
    # Imported from the package rather than hardcoded here: a newly added mode
    # (ab8d) would otherwise silently miss its generator and run as a plain
    # no-augmentation baseline, which looks like a legitimate result.
    from ablation.trainer import BAGAN_MODES as _BAGAN_MODES

    all_results = {}
    cv_summary_table = {}
    cv_start = time.time()

    for mode in MODES_TO_RUN:
        print("=" * 70)
        print(f"§C — Cross-validation: {mode}")
        print("=" * 70)

        # Per-mode deterministic reseed (order-independent CV).
        set_seed(SEED)

        _kwargs = dict(
            config=config,
            mode=mode,
            pool_data=pool_data,
            pool_labels=pool_labels,
            device=device,
        )
        if mode in _BAGAN_MODES:
            _kwargs['bagan_config'] = bagan_config_cv
        elif mode != 'ab1':
            raise RuntimeError(
                f"§C — mode {mode!r} is not in BAGAN_MODES and is not the baseline. "
                f"It would run WITHOUT a generator. Add it to ablation.trainer."
                f"BAGAN_MODES if it needs one."
            )
        if mode in ('ab8c',):
            # ab8c is verbose/plot-heavy inside CV; quiet it down (parity with 124J).
            _kwargs['verbose'] = False
            _kwargs['show_bagan_plots'] = False

        _t = time.time()
        all_results[mode] = run_kfold_ablation(**_kwargs)
        _elapsed = time.time() - _t

        # Record the three CV metrics we report in the final table:
        #   Recall = macro_recall, G-mean = macro_gmean, F2 = macro_f2.
        res = all_results[mode]
        mean_recall, _ = res['cv_summary']['macro_recall']
        mean_gmean,  _ = res['cv_summary']['macro_gmean']
        mean_f2,     _ = res['cv_summary']['macro_f2']
        cv_summary_table[mode] = {
            'cv_recall_mean':     mean_recall,
            'cv_gmean_mean':      mean_gmean,
            'cv_f2_mean':         mean_f2,
            'avg_stopping_epoch': res['avg_stopping_epoch'],
        }
        print(f"\n⏱️  {mode} CV done in {_elapsed:.1f}s "
              f"({_elapsed/60:.1f} min)  folds={len(res['fold_models'])}")
        print(f"   Recall={mean_recall:.4f}  G-mean={mean_gmean:.4f}  F2={mean_f2:.4f}  "
              f"avg_stop_epoch={res['avg_stopping_epoch']}")

    cv_elapsed = time.time() - cv_start
    print("\n" + "=" * 70)
    print(f"✅ §C complete — CV for {len(all_results)} modes in "
          f"{cv_elapsed:.1f}s ({cv_elapsed/60:.1f} min)")
    for mode in MODES_TO_RUN:
        s = cv_summary_table[mode]
        print(f"   {mode:5s}  Recall={s['cv_recall_mean']:.4f}  "
              f"G-mean={s['cv_gmean_mean']:.4f}  F2={s['cv_f2_mean']:.4f}  "
              f"avg_stop_epoch={s['avg_stopping_epoch']}")

else:
    print("[reuse] skipped §C \u2014 REUSE_EXISTING_DATA=True")

In [ ]:
from datetime import datetime
print(f"\n CV time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

import time   # local alias to avoid shadowing if already imported

time_stamp_2 = time.time()

## §D · BAGAN — Retrain-Pool BAGAN Training

A **single shared full-pool BAGAN** is trained here and reused by every BAGAN mode in the §E retrain loop (`ab8a`, `ab8d` on the `bagan` pass; `ab8c` on the `bagan_gp` pass). The raw baseline (`ab1`) ignores it.

> §D.0 pre-splits `pool_data` into `retrain_train_data` (90%) and `retrain_val_data` (10%) **before** BAGAN training. The BAGAN (§D.1–§D.5) trains ONLY on `retrain_train_data`, so the holdout stays clean for §E early stopping.
>
> **Backend budget:** vanilla `bagan` keeps defaults (ae=30/gan=200); `bagan_gp` overrides to ae=50 / gan=300×n_critic so the WGAN-GP critic's inner steps don't starve the generator.
>
> **Shared generator, different selection.** `ab8a` and `ab8d` both consume this one generator and differ only in their acquisition rule — which is why they report identical FID. Only `ab8c` updates the generator (per-cycle co-training), and it receives a private deep copy in §E.1 so the shared model is never mutated.


### §D.0 — Pre-Split Pool (Leakage Prevention)

Split `pool_data` into `retrain_train_data` (90%) and `retrain_val_data` (10%) via `_split_retrain_holdout()` **before** BAGAN training. The BAGAN (§D.1–§D.5) trains ONLY on `retrain_train_data`; the holdout is reserved for §E early stopping.

**Variables created:**
- `retrain_train_data` / `retrain_train_labels` — ~90% of pool (BAGAN + §E classifier training)
- `retrain_val_data` / `retrain_val_labels` — ~10% of pool (§E early stopping only)


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.0 — Pre-Split Pool for Retrain BAGAN (Leakage Prevention)
    # =============================================================================
    #
    # Split pool_data into train/holdout BEFORE training the retrain-pool BAGAN.
    # The BAGAN (§D.1–§D.5) will train ONLY on retrain_train_data.
    # The holdout (retrain_val_data) is reserved for §E early stopping.
    #
    # This prevents the data leakage where BAGAN sees samples that are
    # later used for early-stopping validation in retrain_on_full_pool().
    # =============================================================================

    from ablation.kfold import _split_retrain_holdout

    if config.retrain_val_fraction > 0:
        retrain_train_data, retrain_train_labels, retrain_val_data, retrain_val_labels = \
            _split_retrain_holdout(
                pool_data, pool_labels,
                config.retrain_val_fraction, config.random_seed,
            )
    else:
        retrain_train_data, retrain_train_labels = pool_data, pool_labels
        retrain_val_data = retrain_val_labels = None

    print(f"Pool pre-split for retrain BAGAN:")
    print(f"  Train: {len(retrain_train_data)} samples")
    if retrain_val_data is not None:
        print(f"  Val:   {len(retrain_val_data)} samples (held out from BAGAN)")
        for cls in sorted(retrain_val_labels.unique().tolist()):
            n_train = (retrain_train_labels == cls).sum().item()
            n_val = (retrain_val_labels == cls).sum().item()
            print(f"    Class {cls}: train={n_train}, val={n_val}")

else:
    print("[reuse] skipped §D.0 \u2014 REUSE_EXISTING_DATA=True")

### §D.1 — Full-Pool BAGANConfig & DataLoader

Instantiate the **full-budget** `BAGANConfig` (the §B one is the reduced-epoch copy used inside k-fold CV) and build the BAGAN training `DataLoader` over the 90% retrain split from §D.0. The cell also prints the active minority-FID flags so every run self-documents which configuration produced its FID numbers.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.1 — Full-pool BAGANConfig + BAGAN DataLoader
    # =============================================================================
    # BAGANConfig.from_ablation_config(config) syncs num_classes, latent_dim,
    # device, and the saved/ paths with the active AblationConfig. The backend
    # class (vanilla BAGAN or BAGAN-GP) was bound in §A.1 from GAN_BACKEND.
    #
    # Note: bagan_config_cv (§B) is the *reduced-epoch* config used inside k-fold CV.
    # Here we instantiate the *full* config used for the shared retrain BAGAN —
    # same model/dims, but full ae_epochs / gan_epochs.

    bagan_config = BAGANConfig.from_ablation_config(config)
    bagan_config.save_checkpoints = False  # Disable all disk saves

    # Backend-aware full-pool budget:
    #   - vanilla bagan keeps dataclass defaults (ae_epochs=30, gan_epochs=200).
    #   - bagan_gp gets a Commit-B-equivalent budget so the WGAN-GP critic's
    #     n_critic inner steps do not starve the generator:
    #       gan_epochs = 300 * n_critic outer epochs -> 300 effective G-epochs.
    if GAN_BACKEND == 'bagan_gp':
        _nc_full = getattr(bagan_config, 'n_critic', 1)
        bagan_config.ae_epochs  = 50
        bagan_config.gan_epochs = 300 * _nc_full  # 5 -> 1500
        print(f"  [bagan_gp] Full-pool budget overridden: "
              f"ae_epochs={bagan_config.ae_epochs}, gan_epochs={bagan_config.gan_epochs} "
              f"(=300*n_critic={_nc_full}, effective G-epochs={bagan_config.gan_epochs // _nc_full})")

    print(f"✅ BAGANConfig created  [backend={GAN_BACKEND}]")
    print(f"   num_classes:  {bagan_config.num_classes}")
    print(f"   latent_dim:   {bagan_config.latent_dim}")
    print(f"   batch_size:   {bagan_config.batch_size}")
    print(f"   ae_epochs:    {bagan_config.ae_epochs}")
    print(f"   gan_epochs:   {bagan_config.gan_epochs}")
    print(f"   device:       {bagan_config.device}")

    # ─────────────────────────────────────────────────────────────────────────────
    # Minority-FID fix flags (active only on the bagan_gp backend). Printed here so
    # every run self-documents which configuration produced its FID numbers.
    #   - balanced_critic   : feed the WGAN-GP critic ~33:33:33 real batches
    #   - latent_cov_type   : 'shrinkage' replaces the brittle single scalar σ_c
    #                         with a per-dim σ shrunk toward the isotropic mean
    #   - latent_shrinkage  : λ in var = (1-λ)·var_dim + λ·var_iso  (1=isotropic)
    # vanilla bagan ignores these (defaults to isotropic, unbalanced) → clean baseline.
    print(f"\n   ── minority-FID fixes [{GAN_BACKEND}] ──")
    print(f"   balanced_critic:  {getattr(bagan_config, 'balanced_critic', False)}")
    print(f"   latent_cov_type:  {getattr(bagan_config, 'latent_cov_type', 'isotropic')}")
    print(f"   latent_shrinkage: {getattr(bagan_config, 'latent_shrinkage', 1.0)}")

    # ─────────────────────────────────────────────────────────────────────────────
    # BAGAN Training DataLoader (trains ONLY on the 90% retrain split)
    # ─────────────────────────────────────────────────────────────────────────────
    # Used by AutoencoderTrainer.fit() (§D.3), ClassConditionalLatentGenerator.fit()
    # (§D.4), and GANTrainer.fit() (§D.5).
    bagan_dataset = TensorDataset(retrain_train_data, retrain_train_labels)
    bagan_loader = DataLoader(
        bagan_dataset,
        batch_size=bagan_config.batch_size,
        shuffle=True,     # Randomize batch order each epoch
        drop_last=True,   # Drop last incomplete batch for BatchNorm stability
        num_workers=0,    # Main process (avoids Windows multiprocessing issues)
        pin_memory=True,  # Speed up CPU→GPU transfer when using CUDA
    )

    # ─────────────────────────────────────────────────────────────────────────────
    # Verify DataLoader Output (shape [B,1,64,64], range [-1,1], long labels)
    # ─────────────────────────────────────────────────────────────────────────────
    sample_batch = next(iter(bagan_loader))
    sample_images, sample_labels = sample_batch

    print(f"\n{'─' * 50}")
    print(f"  DataLoader Verification")
    print(f"{'─' * 50}")
    print(f"  Total samples:    {len(bagan_dataset)}")
    print(f"  Num batches:      {len(bagan_loader)} (batch_size={bagan_config.batch_size}, drop_last=True)")
    print(f"  Sample batch:")
    print(f"    Images shape:   {sample_images.shape}")
    print(f"    Images dtype:   {sample_images.dtype}")
    print(f"    Images range:   [{sample_images.min():.3f}, {sample_images.max():.3f}]")
    print(f"    Labels shape:   {sample_labels.shape}")
    print(f"    Labels dtype:   {sample_labels.dtype}")
    print(f"    Labels unique:  {sorted(sample_labels.unique().tolist())}")

    assert sample_images.shape == (bagan_config.batch_size, 1, 64, 64), \
        f"Expected shape ({bagan_config.batch_size}, 1, 64, 64), got {sample_images.shape}"
    assert sample_images.min() >= -1.0 and sample_images.max() <= 1.0, \
        f"Expected range [-1, 1], got [{sample_images.min()}, {sample_images.max()}]"
    assert sample_labels.dtype == torch.long, \
        f"Expected long labels, got {sample_labels.dtype}"

    print(f"\n✅ BAGANConfig created and DataLoader verified.")
    print(f"   Ready for Autoencoder pre-training (§D.3).")
else:
    print("[reuse] skipped §D.1 \u2014 REUSE_EXISTING_DATA=True")

### §D.2 — Visualize First Batch of `bagan_loader`

Inspect the first training batch organised by class-label column (col 0 = T-shirt/top, col 1 = Pullover, col 2 = Dress) to confirm preprocessing is correct.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.2 — Visualize First Batch of bagan_loader (by label column)
    # =============================================================================
    # Layout: one column per class label (0, 1, 2). Each column shows the first N
    # images in the batch belonging to that label. Images denormalized [-1,1]→[0,1].
    # =============================================================================

    import matplotlib.pyplot as plt
    import math

    # 1. Grab first batch
    vis_images, vis_labels = next(iter(bagan_loader))   # [B, 1, 64, 64], [B]

    num_classes = bagan_config.num_classes               # 3
    rows_per_class = 3                                  # rows to show per column

    # 2. Group indices by label
    class_indices = {c: [] for c in range(num_classes)}
    for i, lbl in enumerate(vis_labels.tolist()):
        class_indices[lbl].append(i)
    for c in range(num_classes):
        class_indices[c] = class_indices[c][:rows_per_class]

    actual_rows = max(len(v) for v in class_indices.values())

    # 3. Plot: columns = labels, rows = samples per label
    fig, axes = plt.subplots(
        actual_rows, num_classes,
        figsize=(num_classes * 2.2, actual_rows * 2.2)
    )
    if actual_rows == 1:
        axes = axes[None, :]

    for col, c in enumerate(range(num_classes)):
        class_name = remapped_class_names.get(c, str(c))
        axes[0, col].set_title(f"Label {c}\n({class_name})", fontsize=11, fontweight='bold')
        for row in range(actual_rows):
            ax = axes[row, col]
            if row < len(class_indices[c]):
                img_idx = class_indices[c][row]
                img = vis_images[img_idx, 0]                    # [64, 64]
                img_disp = (img.cpu().numpy() + 1.0) / 2.0     # [-1,1] → [0,1]
                ax.imshow(img_disp, cmap='gray', vmin=0, vmax=1)
                ax.set_xlabel(f"#{img_idx} | lbl={c}", fontsize=7)
            else:
                ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])

    plt.suptitle(
        f"bagan_loader — First Batch (batch_size={bagan_config.batch_size})\n"
        f"Columns = class labels  |  Rows = individual samples",
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.show()

    print(f"\nBatch label distribution:")
    for c in range(num_classes):
        n = (vis_labels == c).sum().item()
        print(f"  Label {c} ({remapped_class_names.get(c, '?')}): {n} samples")

else:
    print("[reuse] skipped §D.2 \u2014 REUSE_EXISTING_DATA=True")

### §D.3 — Autoencoder Pre-training

Pre-train the BAGAN Autoencoder (Encoder → Latent → Decoder) on the retrain pool (MSE loss, Adam, StepLR). Its weights seed the GAN in §D.5. Checkpoint path: `saved/bagan_autoencoder_131J.pth` (or `bagan_gp_autoencoder_131J.pth` on the second pass).


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.3 — Autoencoder Pre-training
    # =============================================================================
    # Train a BAGAN autoencoder (Encoder→Latent→Decoder) on the retrain pool. The
    # trained encoder/decoder weights are later transferred to the GAN generator &
    # discriminator (§D.5), giving the GAN a head-start.
    #
    #   Loss:     MSE between original and reconstructed images
    #   Epochs:   ae_epochs (30 bagan / 50 bagan_gp)
    #   Optimizer/Scheduler: Adam(β₁=0.5) + StepLR (see BAGANConfig)
    #   set_seed(SEED) is called before model creation for reproducibility.
    # =============================================================================

    import time as _time  # local alias

    print("=" * 70)
    print("§D.3 — Autoencoder Pre-training")
    print("=" * 70)

    # 1. Fix randomness BEFORE model creation
    set_seed(SEED)

    # 2. Create the BAGAN Autoencoder
    autoencoder = BAGANAutoencoder(
        in_channels=bagan_config.img_channels,   # 1 for grayscale
        latent_dim=bagan_config.latent_dim       # 128 by default
    )
    print(f"\nAutoencoder created:")
    print(f"  Input shape:  [batch, {bagan_config.img_channels}, {bagan_config.img_size}, {bagan_config.img_size}]")
    print(f"  Latent dim:   {bagan_config.latent_dim}")
    print(f"  Parameters:   {sum(p.numel() for p in autoencoder.parameters()):,}")

    # 3. Create the AutoencoderTrainer
    ae_trainer = AutoencoderTrainer(
        bagan_config,      # hyperparameters (lr_ae, beta1, beta2, etc.)
        autoencoder,       # model to train (moved to GPU internally)
        bagan_loader       # retrain-pool DataLoader
    )

    # 4. Train the Autoencoder (always from scratch here — no checkpoint reuse)
    _ae_ckpt_exists = False
    _train_elapsed = 0.0

    if _ae_ckpt_exists:
        print(f"\n⏭️  Checkpoint found — loading AE from: {bagan_config.autoencoder_path}")
        ae_trainer.load_checkpoint()
        ae_history = ae_trainer.history
        print("   AE training skipped (checkpoint loaded).")
    else:
        _train_start = _time.time()
        ae_history = ae_trainer.fit(
            epochs=bagan_config.ae_epochs,
            save_every=0,        # no checkpointing
            visualize_every=0    # no inline reconstruction plots
        )
        _train_elapsed = _time.time() - _train_start

    # 5. Training Summary
    print("\n" + "=" * 70)
    print("Autoencoder Training Summary")
    print("=" * 70)
    print(f"  Status:            {'Loaded from checkpoint' if _ae_ckpt_exists else 'Trained from scratch'}")
    print(f"  Epochs in history: {len(ae_history['train_loss'])}")
    print(f"  Final train loss:  {ae_history['train_loss'][-1]:.6f}")
    print(f"  Best train loss:   {min(ae_history['train_loss']):.6f} "
          f"(epoch {ae_history['train_loss'].index(min(ae_history['train_loss'])) + 1})")
    if not _ae_ckpt_exists:
        print(f"  Total time:        {_train_elapsed:.1f}s ({_train_elapsed/60:.1f} min)")
    print("=" * 70)

else:
    print("[reuse] skipped §D.3 \u2014 REUSE_EXISTING_DATA=True")

### §D.4 — Latent-Space Fitting & Visualisation

Fit per-class isotropic Gaussians $\mathcal{N}(\mu_c, \sigma_c^2 I)$ in the encoder's latent space (BAGAN's core innovation), then visualise class clusters via PCA and t-SNE to confirm separability.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.4 — Latent Space Fitting (Class-Conditional Distributions)
    # =============================================================================
    # Learn per-class Gaussian distributions in the autoencoder's latent space:
    #     For each class c:  z ~ N(μ_c, σ_c² I)
    # Isotropic covariance (1 scalar σ per class) is the statistically sound choice
    # for the minority classes (~36 samples each after the 90% split).
    #
    #   1. Extract trained encoder from autoencoder (§D.3)
    #   2. Encode all retrain-pool images → latent vectors
    #   3. Group by class, compute (μ_c, σ_c)
    # =============================================================================

    print("=" * 70)
    print("§D.4 — Latent Space Fitting")
    print("=" * 70)

    # 1. Fix randomness for reproducibility
    set_seed(SEED)

    # 2. Extract the trained encoder from the autoencoder
    encoder = ae_trainer.get_encoder()
    print(f"\nEncoder extracted from trained autoencoder")
    print(f"  Encoder type: {type(encoder).__name__}")
    print(f"  Mapping:      [batch, 1, 64, 64] → [batch, {bagan_config.latent_dim}]")

else:
    print("[reuse] skipped §D.4 (fit setup) \u2014 REUSE_EXISTING_DATA=True")

In [ ]:
if not REUSE_EXISTING_DATA:
    # 3. Create the Class-Conditional Latent Generator
    #    Stores per-class (μ_c, σ_c) after fitting. Uses latent_dim, num_classes,
    #    device, and latent_regularization (min σ) from bagan_config.
    latent_gen = ClassConditionalLatentGenerator(bagan_config)

    # 4. Fit class-conditional distributions (always from scratch here)
    _latent_ckpt_exists = False

    if _latent_ckpt_exists:
        print(f"\n⏭️  Latent checkpoint found — loading from: {bagan_config.latent_params_path}")
        latent_gen.load()
        print("   Latent fitting skipped (checkpoint loaded).")
    else:
        # fit(encoder, dataloader): encode all images → group by class →
        # compute per-class μ_c ([128]) and σ_c (isotropic scalar). @torch.no_grad().
        latent_gen.fit(
            encoder=encoder,          # Trained encoder from §D.3
            dataloader=bagan_loader   # Retrain-pool images
        )

else:
    print("[reuse] skipped §D.4 (fit) \u2014 REUSE_EXISTING_DATA=True")

In [ ]:
if not REUSE_EXISTING_DATA:
    # 5. Save fitted parameters (skipped — single-run notebook, kept in memory)
    if False:
        latent_gen.save()
    else:
        print(f"Latent save skipped (kept in memory for §D.5 / §E).")

    # 6. Summary & Verification (per-class stats in remapped contiguous labels)
    print("\n" + "=" * 70)
    print("Latent Space Fitting Summary")
    print("=" * 70)

    stats = latent_gen.get_statistics()
    for cls_label in sorted(stats['n_samples'].keys()):
        cls_name = remapped_class_names.get(cls_label, f"Class {cls_label}")
        n = stats['n_samples'][cls_label]
        sigma = stats['stds'][cls_label]
        # σ_c is a scalar for isotropic, a [latent_dim] vector for shrinkage/diagonal.
        sigma_mean = float(np.mean(sigma))
        sigma_desc = f"σ_c={sigma_mean:.6f}" if np.ndim(sigma) == 0 else f"σ̄_c={sigma_mean:.6f} (vec[{np.size(sigma)}])"
        mu_norm = float(np.linalg.norm(stats['means'][cls_label]))
        orig_label = inverse_label_map.get(cls_label, '?')
        print(f"  Class {cls_label} (orig {orig_label}, {cls_name:>12}): n={n:>5}, {sigma_desc}, ||μ_c||={mu_norm:.4f}")

    print(f"\n  Covariance model: {stats.get('covariance_type', 'isotropic')}")
    print(f"  Fitted:  {latent_gen.is_fitted}")
    print("=" * 70)
else:
    print("[reuse] skipped §D.4 (summary) \u2014 REUSE_EXISTING_DATA=True")

In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.4 (cont.) — Latent Space Visualization
    # =============================================================================
    # Verify: (1) classes form distinct clusters (PCA & t-SNE), (2) fitted Gaussians
    # match the encoded data (comparison plot).

    from bagan import BAGANVisualizer

    visualizer = BAGANVisualizer(config=bagan_config, figsize=(10, 8))

    # 1. Encode all retrain-pool images → latent vectors
    import torch

    encoder.eval()
    all_z = []
    all_y = []
    with torch.no_grad():
        for images, labels in bagan_loader:
            images = images.to(bagan_config.device)
            z = encoder(images)
            all_z.append(z.cpu())
            all_y.append(labels.cpu())

    latent_vectors = torch.cat(all_z, dim=0)
    latent_labels = torch.cat(all_y, dim=0)
    print(f"Encoded {latent_vectors.shape[0]} images → latent dim {latent_vectors.shape[1]}")

    # 2. PCA projection — global structure
    fig_pca, ax_pca = visualizer.plot_latent_space(
        latent_vectors, latent_labels,
        method='pca',
        title='Latent Space — PCA (after AE pre-training)'
    )
    plt.show()

    # 3. t-SNE projection — local neighborhoods
    fig_tsne, ax_tsne = visualizer.plot_latent_space(
        latent_vectors, latent_labels,
        method='tsne',
        title='Latent Space — t-SNE (after AE pre-training)'
    )
    plt.show()

    # 4. Compare real encodings vs. fitted Gaussian samples
    fig_dist, ax_dist = visualizer.plot_class_distributions(
        latent_gen,
        n_samples=500,
        method='pca',
        title='Fitted Class-Conditional Distributions (sampled from N(μ_c, σ_c²I))'
    )
    plt.show()

    print("\n✅ Latent space visualization complete.")
    print("   Classes should show distinct clusters with good separation.")

else:
    print("[reuse] skipped §D.4 (viz) \u2014 REUSE_EXISTING_DATA=True")

### §D.5 — Weight Transfer & GAN Training

Transfer Autoencoder weights to the GAN (decoder → generator all layers; encoder conv → discriminator), then train on the retrain pool. The resulting `generator` / `latent_gen` / `discriminator` are the shared models reused by every BAGAN mode in §E.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.5 — Weight Transfer & GAN Training  (Part 1/3: Model Creation + WT)
    # =============================================================================
    # Create Generator + Discriminator and initialize them with the pre-trained
    # Autoencoder weights from §D.3:
    #     Decoder  → Generator      (ALL layers)
    #     Encoder  → Discriminator  (Conv layers ONLY; classifier head random)
    # set_seed(SEED) before model creation for reproducible init of untransferred
    # layers.

    print("=" * 70)
    print("§D.5 — Weight Transfer & GAN Training")
    print("=" * 70)

    # 1. Fix randomness before model creation
    set_seed(SEED)

    # 2. Create Generator
    generator = BAGANGenerator(
        out_channels=bagan_config.img_channels,  # 1 (grayscale)
        latent_dim=bagan_config.latent_dim       # 128
    )
    print(f"\n📦 Generator created:")
    print(f"   Input:      z ∈ ℝ^{bagan_config.latent_dim} (latent vector)")
    print(f"   Output:     [batch, {bagan_config.img_channels}, {bagan_config.img_size}, {bagan_config.img_size}]")
    print(f"   Activation: Tanh → output range [-1, 1]")
    print(f"   Parameters: {sum(p.numel() for p in generator.parameters()):,}")

    # 3. Create Discriminator
    discriminator = BAGANDiscriminator(
        in_channels=bagan_config.img_channels,  # 1 (grayscale)
        num_classes=bagan_config.num_classes,    # 3 (the +1 fake class is added internally)
        dropout_rate=bagan_config.d_dropout_rate # regularize D to prevent dominance
    )
    print(f"\n📦 Discriminator created:")
    print(f"   Input:      [batch, {bagan_config.img_channels}, {bagan_config.img_size}, {bagan_config.img_size}]")
    print(f"   Output:     [batch, {bagan_config.num_discriminator_outputs}] (logits: {bagan_config.num_classes} real + 1 fake)")
    print(f"   Dropout:    {bagan_config.d_dropout_rate} (between conv blocks)")
    print(f"   Parameters: {sum(p.numel() for p in discriminator.parameters()):,}")

    # 4. Initialize model weights via weight transfer (always — no checkpoint reuse)
    _gan_v1_ckpt_exists = False

    if _gan_v1_ckpt_exists:
        print(f"\n⏭️  GAN checkpoints found — loading weights directly (skipping weight transfer)")
        import torch as _torch
        _ckpt_g = _torch.load(bagan_config.generator_path, map_location=bagan_config.device, weights_only=False)
        _ckpt_d = _torch.load(bagan_config.discriminator_path, map_location=bagan_config.device, weights_only=False)
        generator.load_state_dict(_ckpt_g['model_state_dict'])
        discriminator.load_state_dict(_ckpt_d['model_state_dict'])
        print("   ✅ Model weights loaded from checkpoints.")
        transfer_stats = None
    else:
        #   A) Decoder → Generator  (FULL)
        #   B) Encoder → Discriminator  (Conv layers only; classifier head skipped)
        print("\n" + "─" * 70)
        print("  Transferring Autoencoder Weights → GAN")
        print("─" * 70)
        transfer_stats = WeightTransfer.initialize_gan_from_autoencoder(
            autoencoder=autoencoder,       # Trained in §D.3
            generator=generator,           # Freshly created (random weights)
            discriminator=discriminator,   # Freshly created (random weights)
            verbose=True
        )
        print(f"\n✅ Weight transfer complete!")
        print(f"   Total parameters transferred: {transfer_stats['total_params_transferred']:,}")
        print(f"   Generator:     fully initialized from Decoder")
        print(f"   Discriminator: conv layers from Encoder, classifier head = random")

else:
    print("[reuse] skipped §D.5 (part1) \u2014 REUSE_EXISTING_DATA=True")

In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.5 — GAN Training  (Part 2/3: Train the GAN)
    # =============================================================================
    # D's goal: classify REAL images into true class (0/1/2) AND label FAKE as the
    #           fake class. G's goal: fool D into classifying fakes as a real class.

    set_seed(SEED)

    # 1. Create GANTrainer
    gan_trainer = GANTrainer(
        config=bagan_config,
        generator=generator,
        discriminator=discriminator,
        latent_generator=latent_gen,   # Fitted in §D.4 (is_fitted=True)
        train_loader=bagan_loader      # From §D.1
    )
    print("✅ GANTrainer created successfully")
    print(f"   Generator LR:      {bagan_config.lr_g}")
    print(f"   Discriminator LR:  {bagan_config.lr_d}")

    # 2. Train the GAN
    if _gan_v1_ckpt_exists:
        print(f"\n⏭️  GAN checkpoints loaded in Part 1 — skipping GAN training")
        gan_history = {}
    else:
        print(f"\n🚀 Starting GAN training for {bagan_config.gan_epochs} epochs...")
        print(f"   Device: {bagan_config.device}\n")
        gan_history = gan_trainer.fit(
            epochs=bagan_config.gan_epochs,
            verbose=False,
            save_every=0  # no checkpoint saving
        )
        n_epochs_trained = len(gan_history['d_loss'])
        print(f"\n✅ GAN training complete after {n_epochs_trained} epochs!")

else:
    print("[reuse] skipped §D.5 (part2) \u2014 REUSE_EXISTING_DATA=True")

In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.5 — Training Summary & Verification  (Part 3/3)
    # =============================================================================
    # 1. Training curves   2. Final-epoch metrics   3. Generated-sample grid

    import os
    import numpy as np
    import matplotlib.pyplot as plt

    print("=" * 70)
    print("§D.5 — Training Summary & Verification")
    print("=" * 70)

    # 1. Plot Training Curves
    print("\n📊 Training Curves:")
    if hasattr(gan_trainer, "plot_training_curves"):
        gan_trainer.plot_training_curves()
    else:
        # BAGAN-GP path: plot curves directly from gan_history
        epochs = np.arange(1, len(gan_history.get('d_loss', [])) + 1)
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))

        axes[0].plot(epochs, gan_history.get('d_loss', []), label='D Loss', linewidth=1.8)
        axes[0].plot(epochs, gan_history.get('d_loss_real', []), label='D Loss Real', linewidth=1.4, alpha=0.85)
        axes[0].plot(epochs, gan_history.get('d_loss_fake', []), label='D Loss Fake', linewidth=1.4, alpha=0.85)
        axes[0].plot(epochs, gan_history.get('g_loss', []), label='G Loss', linewidth=1.8)
        axes[0].set_title('Loss Curves'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
        axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)

        axes[1].plot(epochs, gan_history.get('d_acc_real', []), label='D Acc Real', linewidth=1.8)
        axes[1].plot(epochs, gan_history.get('d_acc_fake', []), label='D Acc Fake', linewidth=1.8)
        axes[1].set_title('Discriminator Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
        axes[1].set_ylim(0.0, 1.05); 
        axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)

        d = np.array(gan_history.get('d_loss', []), dtype=float)
        g = np.array(gan_history.get('g_loss', []), dtype=float)
        ratio = np.divide(g, np.abs(d) + 1e-8)
        axes[2].plot(epochs, ratio, label='|G Loss| / |D Loss|', linewidth=1.8)
        axes[2].axhline(1.0, color='gray', linestyle='--', linewidth=1)
        axes[2].set_title('Generator/Discriminator Loss Ratio'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Ratio')
        axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)

        plt.tight_layout()
        plt.show()

    # 2. Final Epoch Metrics
    n_epochs_trained = len(gan_history['d_loss'])
    print(f"\n📋 Final Metrics (epoch {n_epochs_trained}):")
    print(f"   D Loss (total):   {gan_history['d_loss'][-1]:.4f}")
    print(f"   D Loss (real):    {gan_history['d_loss_real'][-1]:.4f}")
    print(f"   D Loss (fake):    {gan_history['d_loss_fake'][-1]:.4f}")
    print(f"   G Loss:           {gan_history['g_loss'][-1]:.4f}")
    print(f"   D Acc (real):     {gan_history['d_acc_real'][-1]:.2%}")
    print(f"   D Acc (fake):     {gan_history['d_acc_fake'][-1]:.2%}")
    if 'wd' in gan_history:
        print(f"   Wasserstein Dist: {gan_history['wd'][-1]:.4f}")
    if 'gp' in gan_history:
        print(f"   Gradient Penalty: {gan_history['gp'][-1]:.4f}")

    # 3. Generate Sample Images per Class
    print(f"\n🖼️  Generated Samples per Class:")
    n_samples_per_class = 4
    fig, axes = plt.subplots(
        bagan_config.num_classes, n_samples_per_class,
        figsize=(n_samples_per_class * 2, bagan_config.num_classes * 2)
    )
    for class_idx in range(bagan_config.num_classes):
        generated = gan_trainer.generate_class_samples(
            target_class=class_idx,
            n_samples=n_samples_per_class
        )
        samples = generated[0] if isinstance(generated, (tuple, list)) else generated
        class_name = remapped_class_names[class_idx]
        for j in range(n_samples_per_class):
            ax = axes[class_idx, j]
            img = samples[j, 0].cpu().detach().numpy()
            img = (img + 1) / 2  # [-1,1] → [0,1]
            ax.imshow(img, cmap='gray')
            ax.axis('off')
            if j == 0:
                ax.set_ylabel(f"Class {class_idx}\n({class_name})", fontsize=9, rotation=0, labelpad=60)
    fig.suptitle(f"Generated Samples after {n_epochs_trained} Epochs", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\n🏁 §D.5 complete! Shared full-pool generator/latent_gen/discriminator ready for §E.")

else:
    print("[reuse] skipped §D.5 (part3) \u2014 REUSE_EXISTING_DATA=True")

### §D.6 — Sample Quality Evaluation *(optional — skip if not needed)*

Measure generated image quality for **all classes** (remapped 0 = T-shirt/top majority, 1 = Pullover, 2 = Dress minorities): diversity (pairwise MSE), sharpness (gradient magnitude), intra-class std, and real/fake pixel similarity. Including the well-sampled majority class gives a reference baseline to compare the augmented minorities against. A per-class FID block follows in the next cell.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.6 — Sample Quality Evaluation
    # =============================================================================
    # Objective quality metrics for ALL classes (remapped 0, 1, 2):
    #   Diversity Score    (pairwise MSE) ↑   — low → mode collapse
    #   Sharpness Score    (gradient magnitude) ↑   — low → blur
    #   Intra-class Std    (pixel-wise std) ↑   — low → low variance
    #   Real-Fake Distance (L1 of mean/std) ↓   — high → statistical mismatch
    # The majority class (0) is included too so its (well-sampled) quality can be
    # compared against the synthetically-augmented minority classes.

    print("=" * 70)
    print("§D.6 — Sample Quality Evaluation")
    print("=" * 70)

    generator.eval()
    eval_classes = list(CLASS_IDS)   # All classes: 0 = T-shirt/top, 1 = Pullover, 2 = Dress

    all_quality_metrics = {}

    for cls in eval_classes:
        orig_label = inverse_label_map[cls]
        cls_name   = remapped_class_names[cls]

        print(f"\n{'─' * 70}")
        print(f"  Class {cls}  (orig {orig_label}, {cls_name})")
        print(f"{'─' * 70}")

        # 1. Gather ALL real samples for this class
        real_mask       = (retrain_train_labels == cls)
        real_images_cls = retrain_train_data[real_mask].to(device)
        n_real          = len(real_images_cls)
        print(f"\n  Real samples:      {n_real}")

        # 2. Generate the same count of fake images from N(μ_c, σ_c²I)
        set_seed(SEED)
        with torch.no_grad():
            z_cls           = latent_gen.sample(class_label=cls, batch_size=n_real)
            fake_images_cls = generator(z_cls)
        print(f"  Generated samples: {len(fake_images_cls)}")

        # 3. Objective quality metrics
        quality_metrics = visualizer.evaluate_sample_quality(
            fake_images=fake_images_cls,
            real_images=real_images_cls,
            n_samples=n_real,
            verbose=True
        )
        all_quality_metrics[cls] = quality_metrics

        # 4. Visual comparison — 8 real vs 8 generated
        fig, axes = visualizer.plot_comparison(
            real_images=real_images_cls,
            fake_images=fake_images_cls,
            n_samples=8,
            title=f"Class {cls} ({cls_name}, orig {orig_label}): Real vs. Generated"
        )
        plt.show()
        plt.close(fig)

    # 5. Side-by-side summary table (majority class 0 first, then minorities)
    print("\n" + "=" * 70)
    print("Quality Metrics Summary — All Classes (majority vs. minority)")
    print("=" * 70)

    col_labels = [f"Class {c} ({remapped_class_names[c]})" for c in eval_classes]
    print(f"{'Metric':<28} " + "  ".join(f"{h:<22}" for h in col_labels))
    print("─" * 70)

    metric_info = [
        ('diversity_score',    'Diversity Score      ↑'),
        ('sharpness_score',    'Sharpness Score      ↑'),
        ('intra_std',          'Intra-class Std      ↑'),
        ('real_fake_distance', 'Real-Fake Distance   ↓'),
    ]
    for key, label in metric_info:
        row = f"{label:<28} "
        for c in eval_classes:
            val = all_quality_metrics[c].get(key, float('nan'))
            row += f"  {val:<22.6f}"
        print(row)

    print("=" * 70)
    print("\n💡 Interpretation thresholds (approximate):")
    print("   Diversity < 0.010 → likely mode collapse")
    print("   Sharpness < 0.050 → images may be blurry")
    print("   Intra-std < 0.100 → low sample variance")
    print("   Real-Fake Dist > 0.30 → statistical mismatch with real data")
    print("=" * 70)

else:
    print("[reuse] skipped §D.6 \u2014 REUSE_EXISTING_DATA=True")

In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.6 (cont.) — Per-Class FID (Fréchet Inception Distance)  ↓ lower is better
    # =============================================================================
    # FID compares InceptionV3 feature statistics of generated vs. real images for
    # each class. It complements the pixel-space metrics above with a perceptual,
    # distribution-level quality score.
    #
    #   - Real reference: per-class real images from the retrain split (cached to
    #     saved/fid_ref_class_<c>_<suffix>.npz so the stats are reused).
    #   - Fake samples:   N_FID_SAMPLES images drawn from N(μ_c, σ_c²I) → generator.
    #   - The InceptionV3 extractor expects [-1,1] tensors (handled internally), so
    #     both real (retrain_train_data) and fake (Tanh output) are passed as-is.
    #
    # ⚠️  Caveat: the minority classes have very few real samples (~36 after the 90%
    #     split), so their real covariance is rank-deficient and the resulting FID is
    #     noisier than the majority class. Treat minority FID as indicative only.
    #
    # 📌 The generator scored here is the WARM-UP generator: it has finished §D.5 GAN
    #    training but has NOT yet been touched by active learning or co-training. It
    #    is the correct reference for attributing a realism change to co-training,
    #    so the triple is persisted to disk at the end of this cell.
    # =============================================================================
    from ablation import build_real_reference_cache, fid_per_class

    N_FID_SAMPLES = 2000   # generated samples per class for the fake-side statistics

    print("=" * 70)
    print("§D.6 (cont.) — Per-Class FID")
    print("=" * 70)

    # 1. Real reference images per class (from the retrain training split).
    real_images_by_class = {
        c: retrain_train_data[retrain_train_labels == c]
        for c in CLASS_IDS
    }
    for c in CLASS_IDS:
        print(f"  Class {c} ({CLASS_NAMES[c]:>12}): {len(real_images_by_class[c])} real reference images")

    # 2. Build (or load) cached real-feature statistics per class.
    set_seed(SEED)
    ref_paths = build_real_reference_cache(
        real_images_by_class=real_images_by_class,
        device=device,
        ref_cache_dir=config.bagan_model_dir,
        date_suffix=config.date_suffix,
    )

    # 3. Compute per-class FID (generated → real).
    set_seed(SEED)
    fid_scores = fid_per_class(
        generator=generator,
        latent_generator=latent_gen,
        classes=CLASS_IDS,
        n_samples=N_FID_SAMPLES,
        device=device,
        real_images_by_class=real_images_by_class,
        ref_cache_dir=config.bagan_model_dir,
        date_suffix=config.date_suffix,
    )

    # 4. Report.
    print("\n" + "=" * 70)
    print(f"Per-Class FID  (n_fake={N_FID_SAMPLES} per class)   ↓ lower is better")
    print("=" * 70)
    print(f"  {'Class':<26} {'#Real':>7}  {'FID':>10}")
    print("─" * 70)
    for c in CLASS_IDS:
        orig = inverse_label_map[c]
        print(f"  Class {c} (orig {orig}, {CLASS_NAMES[c]:>12}) "
              f"{len(real_images_by_class[c]):>7}  {fid_scores[c]:>10.4f}")
    print("─" * 70)
    _minority_fid = [fid_scores[c] for c in CLASS_IDS if c != 0]
    print(f"  Minority-class mean FID: {np.mean(_minority_fid):.4f}")
    print("=" * 70)

    # 5. Persist the WARM-UP generator FID for this backend pass.
    #    Without this the triple exists only in cell output, which is unreachable
    #    once the notebook is executed headlessly by _run_131J_seeds.py. It is
    #    needed because the realism question the paper asks is:
    #
    #        does CO-TRAINING improve sample quality, or is the apparent
    #        improvement just the BAGAN -> BAGAN-GP backbone swap?
    #
    #    Answering that requires comparing §G's co-trained FID against THIS
    #    backend's own warm-up FID. Comparing the BAGAN-GP co-trained generator
    #    against the vanilla-BAGAN column instead conflates the two factors and
    #    overstates the method's contribution.
    _warmup_fid_path = os.path.join(
        config.bagan_model_dir, f"warmup_fid_{config.date_suffix}_{GAN_BACKEND}.npz"
    )
    np.savez(
        _warmup_fid_path,
        classes=np.array(CLASS_IDS),
        fid=np.array([fid_scores[c] for c in CLASS_IDS], dtype=float),
    )
    print(f"\n💾 Saved warm-up generator FID → {_warmup_fid_path}")
    print("   (pre-active-learning reference for the §G co-trained FID)")
else:
    print("[reuse] skipped §D.6 (FID) \u2014 REUSE_EXISTING_DATA=True")

### §D.7 — Autoencoder FID + Sample Dump (per backend)

Two per-backend artifacts that the final §J FID chart and §K sample grid consume:

1. **Per-class autoencoder FID** — `ae_fid_per_class()` decodes class-conditional latent samples through the **AE decoder** and scores them against the same InceptionV3 real-reference cache used in §D.6. One triple `{0,1,2}` per backend pass, saved to `saved/ae_fid_131J_<backend>.npz`.
2. **Sample dump** — 3 fixed images per class for **real**, **autoencoder-decoded**, and **GAN-generated** samples, saved to `saved/samples_131J_<backend>.npz`. The final §K grid merges both passes (BAGAN row from the `bagan` pass, BAGAN-GP row from the `bagan_gp` pass; AE row from the `bagan` pass).


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §D.7 — Autoencoder FID + Sample Dump (shared full-pool models)
    # =============================================================================
    # 1) Per-class AE-FID: decode N(μ_c, σ_c²I) latent samples through the AE decoder
    #    and score vs the §D.6 real-reference cache (saved/fid_ref_class_<c>_131J.npz).
    # 2) Dump 3 fixed real / AE / GAN samples per class for the §K comparison grid.
    #
    # Variables used:    autoencoder (§D.3), latent_gen (§D.4), generator (§D.5),
    #                    real_images_by_class (§D.6), CLASS_IDS, CLASS_NAMES, config
    # Variables created: ae_fid_scores  (+ ae_fid / samples .npz on disk)
    # =============================================================================
    import numpy as np
    from ablation import ae_fid_per_class

    print("=" * 70)
    print(f"§D.7 — Autoencoder FID + Sample Dump  [backend={GAN_BACKEND}]")
    print("=" * 70)

    # ─── 1. Per-class autoencoder FID (reuses the §D.6-warmed real cache) ────────
    set_seed(SEED)
    ae_fid_scores = ae_fid_per_class(
        autoencoder=autoencoder,
        latent_generator=latent_gen,
        classes=CLASS_IDS,
        n_samples=N_FID_SAMPLES,
        device=device,
        real_images_by_class=real_images_by_class,
        ref_cache_dir=config.bagan_model_dir,
        date_suffix=config.date_suffix,
    )
    print(f"\nAutoencoder FID (n_fake={N_FID_SAMPLES} per class)   ↓ lower is better")
    for c in CLASS_IDS:
        print(f"  Class {c} ({CLASS_NAMES[c]:>12}):  AE-FID={ae_fid_scores[c]:8.3f}")

    # Persist the AE-FID triple for this backend pass (consumed by §I / §J).
    _ae_fid_path = os.path.join(config.bagan_model_dir, f"ae_fid_{config.date_suffix}_{GAN_BACKEND}.npz")
    np.savez(_ae_fid_path, classes=np.array(CLASS_IDS),
             ae_fid=np.array([ae_fid_scores[c] for c in CLASS_IDS], dtype=float))
    print(f"\n💾 Saved AE-FID → {_ae_fid_path}")

    # ─── 2. Sample dump: 3 fixed images/class for real, AE-decoded, GAN ──────────
    N_GRID = 3

    @torch.no_grad()
    def _decode_samples(gen_like, classes, k=N_GRID):
        """Draw k class-conditional samples per class from a (latent)->image module."""
        gen_like.eval()
        out = {}
        for c in classes:
            set_seed(SEED + c)  # fixed per-class RNG so the grid is stable across re-runs
            z = latent_gen.sample(class_label=c, batch_size=k).to(device)
            out[c] = gen_like(z).detach().cpu()   # [k,1,64,64] in [-1,1]
        return out

    def _stack_by_class(by_class, classes, k=N_GRID):
        return np.concatenate([by_class[c][:k].numpy() for c in classes], axis=0)  # [k*len,1,64,64]

    _real_by_class = {c: real_images_by_class[c][:N_GRID].cpu() for c in CLASS_IDS}
    _ae_by_class   = _decode_samples(autoencoder.decoder, CLASS_IDS)
    _gen_by_class  = _decode_samples(generator, CLASS_IDS)

    _samples_path = os.path.join(config.bagan_model_dir, f"samples_{config.date_suffix}_{GAN_BACKEND}.npz")
    np.savez(
        _samples_path,
        classes=np.array(CLASS_IDS),
        real=_stack_by_class(_real_by_class, CLASS_IDS),
        ae=_stack_by_class(_ae_by_class, CLASS_IDS),
        gen=_stack_by_class(_gen_by_class, CLASS_IDS),
    )
    print(f"💾 Saved {N_GRID} samples/class (real/ae/gen) → {_samples_path}")
    print(f"\n✅ §D.7 complete — AE-FID + sample dump ready for the final figures.")
    print("=" * 70)

else:
    print("[reuse] skipped §D.7 \u2014 REUSE_EXISTING_DATA=True")

## §E · RETRAIN — Full-Pool Retraining (per mode)

Retrain every mode in `MODES_TO_RUN` on the **retrain-train split** (§D.0's 90% of the pool) for its CV-derived `avg_stopping_epoch`, then hold the result for §F test evaluation. BAGAN modes (`ab8a`, `ab8d`, `ab8c`) reuse the single shared full-pool BAGAN from §D; `ab1` ignores it.

> **Determinism:** each mode is reseeded with a stable, mode-name-derived seed (SHA-256) so its retrained model is **independent of run order / `MODES_TO_RUN` scope**.
>
> **Shared-state safety:** the post-§D BAGAN trio (`generator` / `discriminator` / `latent_gen`) is snapshotted once and restored at the top of every iteration. G-mutating modes (`ab8c`) additionally receive a `deepcopy` so they never alter the shared generator.
>
> **Leakage fix:** the pre-split holdout (`retrain_val_data`, never seen by the §D BAGAN) is passed as `val_data` for best-checkpoint monitoring.
>
> **Watch the generator-drift report at the end of this section.** `ab8c` now runs the per-cycle G-step (131J change), so its ‖w‖ delta should be non-zero. A near-zero delta means the generator was not actually updated and the "co-training" component is inert.


### §E.1 — Retrain on the Retrain-Train Split

Produces **`retrained_models`** — a dict mapping each mode to a `CNNClassifier` trained on `retrain_train_data` for `config.get_retrain_max_epochs(mode, avg_stop)` epochs, with best-checkpoint selection monitored on the `retrain_val_data` holdout (`config.selection_metric`, i.e. **`macro_f2`** — aligned with the §C CV selection metric). Inputs consumed: `cv_summary_table` (§C), `retrain_train_data`/`retrain_val_data` (§D.0), and `generator`/`latent_gen`/`discriminator` (§D.5).


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §E.1 — Retrain Each Mode on the Retrain-Train Split
    # =============================================================================
    # For every mode in MODES_TO_RUN, retrain a fresh CNN classifier on
    # retrain_train_data (§D.0's 90% split) for its CV-derived stopping epoch.
    # The retrain_val_data holdout is passed for best-checkpoint monitoring.
    #
    # Determinism: a stable SHA-256(mode-name) seed pins each mode to the SAME RNG
    # stream regardless of run order / MODES_TO_RUN scope.
    # Shared-state safety: the post-§D BAGAN trio is snapshotted once and restored
    # at the top of each iteration; G-mutating modes (ab8c) get a deepcopy.
    #
    # Variables created:  retrained_models, per_mode_bagan, retrain_audits
    # =============================================================================

    import copy
    import hashlib as _hashlib
    import json as _json
    from pathlib import Path as _Path

    from ablation.kfold import retrain_on_full_pool
    from ablation.data import create_reserve_pool
    from ablation.reproducibility import save_bagan_snapshot, load_bagan_snapshot, sd_hash

    # ─── Preflight: required variables from earlier sections ────────────────────
    _missing_vars = []
    if 'cv_summary_table' not in dir():
        _missing_vars.append('cv_summary_table    ← run §C (CV loop)')
    if 'retrain_train_data' not in dir():
        _missing_vars.append('retrain_train_data  ← run §D.0 (retrain split)')
    if 'generator' not in dir():
        _missing_vars.append('generator           ← run §D.5 (GAN training)')
    if 'latent_gen' not in dir():
        _missing_vars.append('latent_gen          ← run §D.4 (latent fitting)')
    if 'discriminator' not in dir():
        _missing_vars.append('discriminator       ← run §D.5 (GAN training)')
    if _missing_vars:
        raise RuntimeError("§E.1 — missing variables. Run these first:\n  " + "\n  ".join(_missing_vars))

    missing_summary = [m for m in MODES_TO_RUN if m not in cv_summary_table]
    if missing_summary:
        raise KeyError(f"Missing cv_summary_table entries for modes: {missing_summary}. Rerun §C.")

    print("=" * 70)
    print(f"§E.1 — Retrain on Retrain-Train Split  [backend={GAN_BACKEND}]")
    print("=" * 70)
    print(f"  Retrain monitoring hold-out: {config.retrain_val_fraction:.0%}")

    # ─── Reserve pool (only ab1b uses it; not in 125J's modes → None) ───────────
    if 'ab1b' in MODES_TO_RUN:
        reserve_data, reserve_labels = create_reserve_pool(pool_indices)
        print(f"\n📦 Reserve pool: {len(reserve_data)} images")
    else:
        reserve_data = reserve_labels = None

    # ─── Per-mode deterministic reseed (order-independent reproducibility) ──────
    def _mode_retrain_seed(mode_name, base_seed=SEED):
        """Stable per-mode seed via SHA-256 (NOT salted hash())."""
        _h = int(_hashlib.sha256(mode_name.encode('utf-8')).hexdigest(), 16)
        return (base_seed + _h) % (2**31 - 1)

    # ─── Snapshot the post-§D BAGAN trio once; restore before each mode ─────────
    # Backend-specific path so the bagan and bagan_gp passes never collide.
    _bagan_snap_path = f"saved/bagan_snapshot_{config.date_suffix}_{GAN_BACKEND}.pth"
    save_bagan_snapshot(_bagan_snap_path, generator, discriminator, latent_gen)
    print(f"[snapshot] Saved BAGAN snapshot → {_bagan_snap_path}")

    # Modes whose retrain MUTATES the generator (so they need a private deepcopy).
    _G_MUTATING_MODES = ('ab8c', 'ab10', 'ab11', 'ab11a')

    retrained_models = {}
    per_mode_bagan = {}
    retrain_audits = {}
    _g_drift = {}
    total_start = time.time()

    for mode in MODES_TO_RUN:
        # Reseed deterministically per mode BEFORE any stochastic work.
        _mode_seed = _mode_retrain_seed(mode)
        set_seed(_mode_seed)

        # Restore the shared BAGAN trio so no mutable state leaks across modes.
        load_bagan_snapshot(_bagan_snap_path, generator, discriminator, latent_gen)
        _bagan_in_hash = {
            'G': sd_hash(generator.state_dict()),
            'D': sd_hash(discriminator.state_dict()),
            'L': sd_hash({'means': latent_gen.means, 'stds': latent_gen.stds}),
        }

        avg_stop = cv_summary_table[mode]['avg_stopping_epoch']
        max_ep   = config.get_retrain_max_epochs(mode, avg_stop)
        pat      = config.get_retrain_patience(mode)
        print(f"\n{'─' * 60}")
        print(f"  Retraining {mode}: avg_stop={avg_stop}, max_epochs={max_ep}, "
              f"patience={pat}, seed={_mode_seed}")

        # G-mutating modes get a private deepcopy so the shared generator is safe.
        if mode in _G_MUTATING_MODES:
            _gen = copy.deepcopy(generator)
            _lat = copy.deepcopy(latent_gen)
            _gen_norm_before = sum(p.data.norm().item() for p in _gen.parameters())
            print(f"    ℹ️  {mode}: deep-copied generator (param ‖w‖={_gen_norm_before:.2f})")
        else:
            _gen = generator
            _lat = latent_gen
            _gen_norm_before = None

        mode_start = time.time()
        retrained_models[mode] = retrain_on_full_pool(
            config=config,
            mode=mode,
            pool_data=retrain_train_data,
            pool_labels=retrain_train_labels,
            avg_stopping_epoch=avg_stop,
            device=device,
            generator=_gen,
            latent_generator=_lat,
            discriminator=discriminator,
            reserve_data=reserve_data,
            reserve_labels=reserve_labels,
            val_data=retrain_val_data,
            val_labels=retrain_val_labels,
            verbose=True,
        )
        mode_elapsed = time.time() - mode_start

        # Keep the BAGAN used by this mode (None for non-BAGAN baselines).
        per_mode_bagan[mode] = None if mode == 'ab1' else {'generator': _gen, 'latent_gen': _lat}

        # Per-mode audit: reseed + BAGAN in/out hashes for reproducibility checks.
        retrain_audits[mode] = dict(getattr(retrained_models[mode], '_retrain_audit', {}))
        retrain_audits[mode]['retrain_seed']  = _mode_seed
        retrain_audits[mode]['bagan_in_hash'] = _bagan_in_hash
        retrain_audits[mode]['bagan_out_hash'] = {
            'G': sd_hash(generator.state_dict()),
            'D': sd_hash(discriminator.state_dict()),
            'L': sd_hash({'means': latent_gen.means, 'stds': latent_gen.stds}),
        }

        # Track generator drift for G-mutating modes (sanity check).
        if mode in _G_MUTATING_MODES:
            _gen_norm_after = sum(p.data.norm().item() for p in _gen.parameters())
            _delta = abs(_gen_norm_after - _gen_norm_before)
            _flag = '✓ G mutated' if _delta > 1e-6 else '⚠️ G may not have updated'
            print(f"    ℹ️  {mode}: G param ‖w‖ after={_gen_norm_after:.2f}  Δ={_delta:.4f}  {_flag}")
            _g_drift[mode] = {'norm_before': _gen_norm_before, 'norm_after': _gen_norm_after,
                              'delta': _delta, 'n_epochs': avg_stop}

        print(f"  {mode} retrained ✓  ({mode_elapsed:.1f}s)")

    total_elapsed = time.time() - total_start

    # ─── Summary ────────────────────────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"  §E.1 Retrain Summary  [backend={GAN_BACKEND}]")
    print(f"{'=' * 60}")
    for mode, model in sorted(retrained_models.items()):
        avg_stop = cv_summary_table[mode]['avg_stopping_epoch']
        max_ep   = config.get_retrain_max_epochs(mode, avg_stop)
        pat      = config.get_retrain_patience(mode)
        print(f"  {mode:6s}  avg_stop={avg_stop:3d}  max_ep={max_ep:3d}  "
              f"patience={pat:2d}  model={type(model).__name__}")
    print(f"\n⏱️  Total retrain time: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
    print(f"   Models retrained: {len(retrained_models)}/{len(MODES_TO_RUN)}")

    assert len(retrained_models) == len(MODES_TO_RUN), \
        f"Expected {len(MODES_TO_RUN)} retrained models, got {len(retrained_models)}"

    if _g_drift:
        print(f"\n📊 Generator drift after retrain (G-mutating modes):")
        for _m, _d in _g_drift.items():
            _pct = _d['delta'] / max(_d['norm_before'], 1e-12) * 100
            _verdict = '✓ healthy' if 0.01 <= _pct <= 10 else (
                '⚠️ large (>10%)' if _pct > 10 else '⚠️ near-zero — G may not have updated')
            print(f"   {_m:6s}  ‖w‖ {_d['norm_before']:.2f} → {_d['norm_after']:.2f}  "
                  f"Δ={_d['delta']:.4f} ({_pct:.2f}%)  {_verdict}")

    # ─── Persist audit JSON (backend-specific) for cross-run hash comparison ────
    _audit_path = _Path(config.bagan_model_dir) / f"retrain_audits_{config.date_suffix}_{GAN_BACKEND}.json"
    _audit_path.parent.mkdir(parents=True, exist_ok=True)
    with open(_audit_path, 'w', encoding='utf-8') as _f:
        _json.dump({
            'backend': GAN_BACKEND,
            'modes_to_run': list(MODES_TO_RUN),
            'random_seed': config.random_seed,
            'audits': retrain_audits,
        }, _f, indent=2, default=str)
    print(f"\n💾 Retrain audit written to {_audit_path}")

    print(f"\n✅ §E.1 complete — retrained_models ({len(retrained_models)} modes) ready for §F.")
    print("=" * 70)

else:
    print("[reuse] skipped §E.1 \u2014 REUSE_EXISTING_DATA=True")

## §F — Held-Out Test Evaluation

Evaluate each retrained model from §E.1 on the **held-out test set** (`test_loader`). Metrics are reported with the same **macro** convention used in §C/§E — Recall = `macro_recall`, G-mean = `macro_gmean`, F2 = `macro_f2` — so CV-selection and final-test reporting use one consistent metric family. `evaluate_on_test` also returns binary (minority-detection) and per-class metrics, which are kept in `test_results[mode]` for §G/§H.

Produces **`test_results`** — a dict mapping each mode to its full test-metrics dict.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §F — Held-Out Test Evaluation for All Modes
    # =============================================================================
    # For each mode, evaluate its §E.1 retrained model on the held-out test set.
    # Metrics use the macro convention (macro_recall / macro_gmean / macro_specificity
    # / macro_f2) to stay aligned with §C (CV selection) and §E (retrain checkpoint
    # selection).
    #
    # Variables used:    retrained_models, test_loader, config, device
    # Variables created: test_results — dict[mode] → metrics dict
    # =============================================================================

    from ablation import evaluate_on_test

    # ─── Preflight ──────────────────────────────────────────────────────────────
    if 'retrained_models' not in dir() or not retrained_models:
        raise RuntimeError("§F — retrained_models missing/empty. Run §E.1 first.")

    print("=" * 70)
    print(f"§F — Held-Out Test Evaluation (Retrained Models)  [backend={GAN_BACKEND}]")
    print("=" * 70)
    print(f"  Selection/report metric: {config.selection_metric}  (macro family)")

    # Human-readable labels for every mode the two backend passes can produce.
    # Paper Table 4 uses: Baseline=ab1, Variant1=ab8a, Variant2=ab8d, Proposed=ab8c.
    MODE_LABELS = {
        'ab1':  'Baseline   — Ab-1  (Imbalanced, no augmentation)',
        'ab1f': 'Ab-1f      (Imbalanced baseline, final-epoch)',
        'ab8a': 'Variant 1  — Ab-8a (GAAL, label gate + margin)',
        'ab8d': 'Variant 2  — Ab-8d (GAAL, confidence-gated diverse + latent-SMOTE)',
        'ab8c': 'Proposed   — Ab-8c (confidence-gated + latent-SMOTE + D/G/C co-training)',
    }

    test_results = {}

    for mode in MODES_TO_RUN:
        model = retrained_models[mode]
        test_results[mode] = evaluate_on_test(
            model=model,
            test_loader=test_loader,
            config=config,
            device=device,
        )

        r = test_results[mode]
        label = MODE_LABELS.get(mode, mode)
        print(f"\n  {label}:")
        print(f"    Recall={r['macro_recall']:.4f}  G-mean={r['macro_gmean']:.4f}  "
              f"Specificity={r['macro_specificity']:.4f}  F2={r['macro_f2']:.4f}  "
              f"Precision={r['macro_precision']:.4f}  Acc={r['accuracy']:.4f}")

    print(f"\n✅ §F complete — test_results has {len(test_results)} modes ready for §G/§H.")
    print("=" * 70)

else:
    print("[reuse] skipped §F \u2014 REUSE_EXISTING_DATA=True")

## §G — Per-Class FID for the Final Table

Compute a **per-(mode, class) FID** using each mode's retrained BAGAN snapshot (`per_mode_bagan` from §E.1). FID compares InceptionV3 feature statistics of generated vs. real images — lower is better. The real-feature statistics warmed in §D.6 (`saved/fid_ref_class_<c>_131J.npz`) are reused, so this only computes the fake-side features per mode.

- The non-GAN baseline (`ab1`) has no generator → its FID rows are `NaN`.
- `ab8a` and `ab8d` share the §D generator and therefore report **identical FID** — they differ in selection, not generation. This is expected, not a bug.
- `ab8c` co-trains its own generator copy, so it is the only mode whose FID can differ.

Produces **`fid_df`** — a tidy DataFrame (`mode`, `class_id`, `class_name`, `fid`, `n_real`, `n_fake`) consumed by §H/§I.

> ⚠️ Minority classes have very few real samples (~36 after the 90% split), so their real covariance is rank-deficient and the FID is noisier than the majority class — treat minority FID as indicative only.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §G — Per-Class FID per Mode (for the final ablation table)
    # =============================================================================
    # For each mode, draw N_FID_SAMPLES synthetic images per class from its §E.1
    # retrained BAGAN snapshot (per_mode_bagan[mode]) and score the InceptionV3 FID
    # against the real reference statistics cached in §D.6. Non-GAN baselines
    # (ab1, ab1f) have no generator → NaN rows.
    #
    # Reuses the warmed real-feature cache (saved/fid_ref_class_<c>_131J.npz), so
    # only the fake-side features are computed here.
    #
    # Variables used:    per_mode_bagan, retrain_train_data/labels, CLASS_IDS,
    #                    CLASS_NAMES, N_FID_SAMPLES, config, device
    # Variables created: fid_df — tidy DataFrame[mode, class_id, class_name, fid,
    #                    n_real, n_fake]
    # =============================================================================
    import numpy as np
    import pandas as pd

    from ablation import fid_per_class

    # ─── Preflight ──────────────────────────────────────────────────────────────
    if 'per_mode_bagan' not in dir() or not per_mode_bagan:
        raise RuntimeError("§G — per_mode_bagan missing/empty. Run §E.1 first.")

    # Non-GAN baselines never produce synthetic samples → FID is undefined (NaN).
    _NON_GAN_MODES = {'ab1', 'ab1f'}

    print("=" * 70)
    print(f"§G — Per-Class FID  [backend={GAN_BACKEND}]   ↓ lower is better")
    print(f"     N_FID_SAMPLES per class = {N_FID_SAMPLES}")
    print("=" * 70)

    # Per-class real sample counts (for reporting alongside FID).
    _n_real_by_class = {
        c: int((retrain_train_labels == c).sum().item())
        for c in CLASS_IDS
    }
    for c in CLASS_IDS:
        print(f"  Class {c} ({CLASS_NAMES[c]:>12}): {_n_real_by_class[c]} real reference images")

    fid_rows = []
    for mode in MODES_TO_RUN:
        snap = per_mode_bagan.get(mode)

        # Non-GAN baseline: no generator → NaN FID rows.
        if snap is None or mode in _NON_GAN_MODES:
            for c in CLASS_IDS:
                fid_rows.append({
                    'mode':       mode,
                    'class_id':   c,
                    'class_name': CLASS_NAMES[c],
                    'fid':        float('nan'),
                    'n_real':     _n_real_by_class[c],
                    'n_fake':     0,
                })
            print(f"\n  {mode:6s}  (no BAGAN — FID = NaN for all classes)")
            continue

        gen = snap['generator'].eval().to(device)
        lat = snap['latent_gen']

        # Deterministic fake sampling; reuses cached real stats per class.
        set_seed(SEED)
        mode_fid = fid_per_class(
            generator=gen,
            latent_generator=lat,
            classes=CLASS_IDS,
            n_samples=N_FID_SAMPLES,
            device=device,
            real_images_by_class=None,   # use the §D.6-warmed cache
            ref_cache_dir=config.bagan_model_dir,
            date_suffix=config.date_suffix,
        )

        print(f"\n  {mode}:")
        for c in CLASS_IDS:
            fid_rows.append({
                'mode':       mode,
                'class_id':   c,
                'class_name': CLASS_NAMES[c],
                'fid':        float(mode_fid[c]),
                'n_real':     _n_real_by_class[c],
                'n_fake':     N_FID_SAMPLES,
            })
            print(f"    class={c} ({CLASS_NAMES[c]:>12})  FID={mode_fid[c]:8.3f}")

    fid_df = pd.DataFrame(fid_rows)

    print("\n" + "=" * 70)
    print("Per-Class FID Table  (fid_df)")
    print("=" * 70)
    print(fid_df.to_string(index=False))
    print(f"\n✅ §G complete — fid_df has {len(fid_df)} rows "
          f"({len(MODES_TO_RUN)} modes × {len(CLASS_IDS)} classes).")
    print("=" * 70)

else:
    print("[reuse] skipped §G \u2014 REUSE_EXISTING_DATA=True")

## §H — Collect Rows & Save Per-Backend CSV

Assemble one row per mode for **this backend pass**, combining the result families into a single tidy table:

- **CV (§C)** — `cv_recall`, `cv_gmean`, `cv_f2`, `avg_stopping_epoch` (the selection metrics).
- **Test (§F)** — `test_recall`, `test_gmean`, `test_specificity`, `test_f2`, `test_accuracy` (macro family, held-out).
- **FID (§G)** — per-class `fid_c<k>` plus `fid_minority_mean` (NaN for `ab1`).
- **AE-FID (§D.7)** — per-backend `ae_fid_c<k>` (same triple repeated on every row of this backend).

Each row is tagged with `backend`, a human-readable `variant` label, and the `selector` actually used, so the two passes (`bagan` → `ab1`,`ab8a`,`ab8d`; `bagan_gp` → `ab8c`) can be merged in §I without ambiguity. Recording `selector` on the row is deliberate: it makes the acquisition rule behind each number auditable from the CSV alone rather than from notebook prose. Saved to **`saved/ablation_rows_131J_<backend>.csv`**.


In [ ]:
if not REUSE_EXISTING_DATA:
    # =============================================================================
    # §H — Collect Per-Mode Rows & Save Backend CSV
    # =============================================================================
    # Merge CV (§C), test (§F, incl. macro_specificity), per-mode FID (§G), the
    # warm-up generator FID (§D.6) and the per-backend autoencoder FID (§D.7) into
    # one row per mode, then persist to
    # saved/ablation_rows_<suffix>_<backend>.csv so §I — and the multi-seed
    # aggregator — can merge every pass.
    #
    # The AE-FID and warm-up-FID triples are per-backend (one shared AE and one
    # shared warm-up generator per pass), so they are attached as identical
    # ae_fid_c{0,1,2} / warmup_fid_c{0,1,2} columns on every row of this backend.
    #
    # ⚠️  COLUMN NAMES ARE A CONTRACT with _aggregate_131J_seeds.py. The held-out
    #     metrics are written here as test_*; that script's METRICS list must use
    #     the same names. A mismatch does NOT raise — build_matrix() skips a column
    #     it cannot find — so it silently yields an all-NaN Table 4. Change both
    #     sides together. (This exact drift was found and fixed once already.)
    #
    # Variables used:    MODES_TO_RUN, GAN_BACKEND, cv_summary_table, test_results,
    #                    fid_df, ae_fid_scores, fid_scores, CLASS_IDS, config
    # Variables created: rows_df  (+ CSV on disk)
    # =============================================================================
    import numpy as np
    import pandas as pd
    from pathlib import Path

    # ─── Preflight ──────────────────────────────────────────────────────────────
    # fid_scores is the §D.6 warm-up generator triple; ae_fid_scores is §D.7's.
    for _name in ('cv_summary_table', 'test_results', 'fid_df', 'ae_fid_scores', 'fid_scores'):
        if _name not in dir():
            raise RuntimeError(f"§H — '{_name}' missing. Run §C / §D.6 / §D.7 / §F / §G first.")

    print("=" * 70)
    print(f"§H — Collect Rows & Save CSV  [backend={GAN_BACKEND}]")
    print("=" * 70)

    # Human-readable variant label per (backend, mode) — disambiguates a mode that
    # is run under more than one backend.
    VARIANT_LABELS = {
        ('bagan',    'ab1'):  'Ab-1 (Imbalanced baseline)',
        ('bagan',    'ab1f'): 'Ab-1f (Baseline, final-epoch)',
        ('bagan',    'ab4'):  'Ab-4 (BAGAN Static)',
        ('bagan',    'ab8'):  'Ab-8 (GAAL with margin)',
        ('bagan',    'ab8a'): 'Ab-8a (GAAL, label gate + margin)',
        ('bagan',    'ab8b'): 'Ab-8b (GAAL, BADGE)',
        ('bagan',    'ab8d'): 'Ab-8d (GAAL, confidence-gated)',
        ('bagan',    'ab8c'): 'Ab-8c (BAGAN, confidence-gated + 3-player)',
        ('bagan_gp', 'ab4'):  'Ab-4 (BAGAN-GP Static)',
        ('bagan_gp', 'ab8b'): 'Ab-8b (BAGAN-GP, BADGE)',
        ('bagan_gp', 'ab8d'): 'Ab-8d (BAGAN-GP, confidence-gated)',
        ('bagan_gp', 'ab8c'): 'Ab-8c (confidence-gated + 3-player, proposed)',
    }

    # Wide per-class FID lookup: (mode, class_id) → fid.
    _fid_lookup = {
        (r['mode'], int(r['class_id'])): r['fid']
        for _, r in fid_df.iterrows()
    }

    rows = []
    for mode in MODES_TO_RUN:
        cv = cv_summary_table[mode]
        tr = test_results[mode]

        # Acquisition rule actually used by this mode (blank for non-AL modes).
        try:
            _sel = config.get_selection_params(mode)
            _sel_desc = f"{_sel['mode']}(tau={_sel['min_confidence']})" \
                if _sel['mode'] == 'confident_diverse' else _sel['mode']
        except (KeyError, AttributeError):
            _sel_desc = 'margin' if mode == 'ab8a' else ''

        row = {
            'backend':            GAN_BACKEND,
            'mode':               mode,
            'variant':            VARIANT_LABELS.get((GAN_BACKEND, mode), mode),
            'selector':           _sel_desc,
            # ── CV (§C) ──────────────────────────────────────────────────
            'cv_recall':          cv['cv_recall_mean'],
            'cv_gmean':           cv['cv_gmean_mean'],
            'cv_f2':              cv['cv_f2_mean'],
            'avg_stopping_epoch': cv['avg_stopping_epoch'],
            # ── Test (§F) — macro family incl. specificity ───────────────
            'test_precision':     tr['macro_precision'],
            'test_recall':        tr['macro_recall'],
            'test_gmean':         tr['macro_gmean'],
            'test_specificity':   tr['macro_specificity'],
            'test_f2':            tr['macro_f2'],
            'test_accuracy':      tr['accuracy'],
        }

        # ── Per-mode FID (§G): per-class columns + minority mean ─────────
        _minority_fids = []
        for c in CLASS_IDS:
            _fid = _fid_lookup.get((mode, c), float('nan'))
            row[f'fid_c{c}'] = _fid
            if c != 0:  # class 0 = majority; minorities are the augmented ones
                _minority_fids.append(_fid)
        row['fid_minority_mean'] = float(np.nanmean(_minority_fids)) if _minority_fids else float('nan')

        # ── Autoencoder FID (§D.7): per-backend triple, same on every row ─
        for c in CLASS_IDS:
            row[f'ae_fid_c{c}'] = float(ae_fid_scores[c])

        # ── Warm-up generator FID (§D.6): per-backend triple, same on every row.
        #    This is the PRE-active-learning generator. Recording it per row is what
        #    makes the realism attribution auditable from the CSV alone: the
        #    co-training effect is fid_c* minus warmup_fid_c* WITHIN a backend, not
        #    the difference between two different backends' fid_c*.
        for c in CLASS_IDS:
            row[f'warmup_fid_c{c}'] = float(fid_scores[c])
        row['warmup_fid_minority_mean'] = float(
            np.nanmean([fid_scores[c] for c in CLASS_IDS if c != 0])
        )

        rows.append(row)

    rows_df = pd.DataFrame(rows)

    print("\nCollected rows:")
    print(rows_df.to_string(index=False))

    # ─── Persist backend-specific CSV ───────────────────────────────────────────
    _csv_path = Path(config.bagan_model_dir) / f"ablation_rows_{config.date_suffix}_{GAN_BACKEND}.csv"
    _csv_path.parent.mkdir(parents=True, exist_ok=True)
    rows_df.to_csv(_csv_path, index=False)
    print(f"\n💾 Saved {len(rows_df)} rows → {_csv_path}")

    print(f"\n✅ §H complete — run the OTHER backend pass, then §I merges both CSVs.")
    print("=" * 70)
else:
    print("[reuse] skipped §H \u2014 REUSE_EXISTING_DATA=True")

## §I — Merge Both Backends → Final Ablation Table

The capstone merge cell. **Run after the backend passes you want have written their CSVs** (`ablation_rows_131J_bagan.csv` and/or `ablation_rows_131J_bagan_gp.csv`). It:

1. **Discovers & concatenates** every `saved/ablation_rows_131J_*.csv` into one frame.
2. **Orders** rows canonically — baseline → margin → BADGE → confidence-gated → proposed — keeping only the modes actually run.
3. **Saves** the merged table to **`saved/ablation_summary_131J.csv`** (full precision, every mode run) and prints a paper-ready view with Specificity and the AE-FID columns.
4. **Emits the 4-row publication table** (`saved/ablation_publication_131J.csv`) that becomes Table 4: Baseline → Variant 1 (margin) → Variant 2 (confidence-gated) → Proposed.
5. **Prints a monotonicity check** on the four rows, and the **BADGE vs confidence-gated comparison** on the shared BAGAN backbone.

> **Why the monotonicity print exists.** The four rows are *not* monotone on F2, so the paper must not assert "every component improves performance" — check the printed flags before writing §4.4.

> Disk-only — works across kernel restarts. The visualization cells (§J metrics, §J FID, §K sample grid) read `summary_df` / the saved `.npz` dumps produced here and in §D.7.


In [ ]:
# =============================================================================
# §I — Merge Both Backends → Final Ablation Table
# =============================================================================
# Disk-only capstone: concatenate every saved/ablation_rows_131J_*.csv, order in
# canonical paper sequence, save the merged summary, and print a paper-ready view
# (incl. Specificity + AE-FID). Visualization happens in §J / §K.
#
# Variables created: summary_df, publication_df
#   + ablation_summary_131J.csv
#   + ablation_publication_131J.csv
# =============================================================================
import numpy as np
import pandas as pd
from pathlib import Path

_save_dir = Path(config.bagan_model_dir)
_pattern  = f"ablation_rows_{config.date_suffix}_*.csv"
# Publication-subset outputs (merged summary + Table-4 CSV) are written under
# a distinct "<suffix>_pub4" tag so re-deriving them from already-computed
# per-backend rows CSVs never overwrites the canonical ablation_summary_
# {config.date_suffix}.csv that other notebooks (e.g. the five-seed aggregator)
# may also read.
_pub_suffix = f"{config.date_suffix}_pub4"
_csv_files = sorted(_save_dir.glob(_pattern))

print("=" * 70)
print(f"§I — Merge & Final Table  [pattern={_pattern}]")
print("=" * 70)

if not _csv_files:
    raise FileNotFoundError(
        f"§I — no per-backend CSVs found in {_save_dir} matching '{_pattern}'. "
        f"Run §H for at least one backend pass first."
    )

print(f"\nFound {len(_csv_files)} backend CSV(s):")
for _f in _csv_files:
    print(f"  • {_f.name}")

# ─── Concatenate ────────────────────────────────────────────────────────────
summary_df = pd.concat([pd.read_csv(_f) for _f in _csv_files], ignore_index=True)

_backends_present = set(summary_df['backend'].unique())
if _backends_present != {'bagan', 'bagan_gp'}:
    print(f"\n⚠️  Only backends {sorted(_backends_present)} present — table is PARTIAL. "
          f"Run the missing backend pass for the complete table.")

# ─── Canonical paper ordering: (backend, mode) → rank ───────────────────────
_ORDER = {
    ('bagan',    'ab1'):  0,
    ('bagan',    'ab1f'): 1,
    ('bagan',    'ab4'):  2,
    ('bagan',    'ab8'):  3,
    ('bagan',    'ab8a'): 4,
    ('bagan',    'ab8b'): 5,
    ('bagan',    'ab8d'): 6,
    ('bagan',    'ab8c'): 7,
    ('bagan_gp', 'ab4'):  8,
    ('bagan_gp', 'ab8b'): 9,
    ('bagan_gp', 'ab8d'): 10,
    ('bagan_gp', 'ab8c'): 11,
}
summary_df['_rank'] = summary_df.apply(
    lambda r: _ORDER.get((r['backend'], r['mode']), 99), axis=1
)
summary_df = summary_df.sort_values('_rank').drop(columns='_rank').reset_index(drop=True)

# ─── Save merged summary (full precision) ───────────────────────────────────
# NOTE: this ALWAYS contains every mode that was run, including ab8b (BADGE),
# even though BADGE is not one of the four rows of the paper's Table 4. The
# merged summary is the record of what was actually executed; the publication
# subset below is only a presentation choice.
_summary_path = _save_dir / f"ablation_summary_{_pub_suffix}.csv"
summary_df.to_csv(_summary_path, index=False)
print(f"\n💾 Saved merged summary ({len(summary_df)} rows) → {_summary_path}")

# ─── Paper-ready full view ───────────────────────────────────────────────────
_fid_cols    = [c for c in summary_df.columns if c.startswith('fid_c')]
_ae_fid_cols = [c for c in summary_df.columns if c.startswith('ae_fid_c')]
_display_cols = (
    ['variant', 'selector', 'cv_recall', 'cv_gmean', 'cv_f2', 'avg_stopping_epoch',
     'test_recall', 'test_gmean', 'test_specificity', 'test_f2', 'test_accuracy']
    + _fid_cols + ['fid_minority_mean'] + _ae_fid_cols
)
_display_cols = [c for c in _display_cols if c in summary_df.columns]
_paper = summary_df[_display_cols].copy()
_fmt = {c: '{:.4f}' for c in _paper.columns
        if c not in ('variant', 'selector', 'avg_stopping_epoch')}
_fmt['avg_stopping_epoch'] = '{:.1f}'

print("\n" + "=" * 70)
print(f"FINAL ABLATION TABLE ({config.date_suffix}) — ALL MODES RUN")
print("=" * 70)
with pd.option_context('display.max_columns', None, 'display.width', 260):
    print(_paper.to_string(index=False, formatters={k: v.format for k, v in _fmt.items()}))

# ─── Publication subset (paper Table 4: four variants) ──────────────────────
# Baseline  -> (bagan,    ab1)   imbalanced, no augmentation
# Variant1  -> (bagan,    ab8a)  aux-head label gate + margin selection            (310's "Variant 1")
# Variant2  -> (bagan_gp, ab8d)  confidence-gated diverse selection, GP backbone   (310's "Variant 3")
# Proposed  -> (bagan_gp, ab8c)  Variant2's rule + co-training                     (310's "Proposed")
#
# Variant2 moved from (bagan, ab8d) to (bagan_gp, ab8d): the four-row Table 4 now
# isolates the GP backbone + confidence gate together, with Proposed's co-training
# arrow the only remaining difference between Variant2 and Proposed.
#
# ab8b (BADGE) is RUN and kept in summary_df above, but is not a row of
# Table 4: the paper presents the component chain, and BADGE is an alternative
# acquisition rule rather than a component of the proposed method. Its numbers
# remain available in ablation_summary_{config.date_suffix}.csv and are reported
# in the text.
_PUB_VARIANTS = [
    ('Baseline', 'bagan',    'ab1'),
    ('Variant1', 'bagan',    'ab8a'),
    ('Variant2', 'bagan_gp', 'ab8d'),
    ('Proposed', 'bagan_gp', 'ab8c'),
]

_pub_rows = []
_missing_pub = []
for _pub_name, _backend, _mode in _PUB_VARIANTS:
    _sel = summary_df[(summary_df['backend'] == _backend) & (summary_df['mode'] == _mode)]
    if _sel.empty:
        _missing_pub.append(f"{_pub_name} -> ({_backend}, {_mode})")
        continue
    _r = _sel.iloc[0]
    _pub_rows.append({
        'Variant': _pub_name,
        'Recall': float(_r.get('test_recall', np.nan)),
        'G-mean': float(_r.get('test_gmean', np.nan)),
        'F2': float(_r.get('test_f2', np.nan)),
        'Specificity': float(_r.get('test_specificity', np.nan)),
    })


In [ ]:

publication_df = pd.DataFrame(_pub_rows, columns=['Variant', 'Recall', 'G-mean', 'F2', 'Specificity'])

print("\n" + "=" * 70)
print("PUBLICATION TABLE (4 variants — paper Table 4)")
print("=" * 70)
if publication_df.empty:
    print("⚠️ Publication table is empty. Required rows were not found in summary_df.")
else:
    _fmt_pub = {c: '{:.4f}'.format for c in ['Recall', 'G-mean', 'F2', 'Specificity']}
    print(publication_df.to_string(index=False, formatters=_fmt_pub))

    # Monotonicity check. The paper should not assert "every component helps"
    # unless this actually holds - print it rather than assume it.
    print("\nMonotonicity across the four rows:")
    for _m in ['Recall', 'G-mean', 'F2', 'Specificity']:
        _v = publication_df[_m].tolist()
        _mono = all(_b >= _a for _a, _b in zip(_v, _v[1:]))
        _flag = 'monotone' if _mono else 'NOT monotone'
        print(f"  {_m:<12} {[round(x, 4) for x in _v]}  {_flag}")

    # BADGE was run but is not a Table-4 row; surface it so the comparison is
    # never lost track of, and so the text can quote it accurately.
    # NOTE: pulled straight from summary_df (not publication_df) so this stays a
    # same-backbone (BAGAN) comparison even though Variant2 now lives on bagan_gp.
    _badge = summary_df[(summary_df['backend'] == 'bagan') & (summary_df['mode'] == 'ab8b')]
    _confgated_bagan = summary_df[(summary_df['backend'] == 'bagan') & (summary_df['mode'] == 'ab8d')]
    if not _badge.empty and not _confgated_bagan.empty:
        _b = _badge.iloc[0]
        _g = _confgated_bagan.iloc[0]
        print("\nAcquisition-rule comparison (same BAGAN backbone, run but not in Table 4):")
        print(f"  {'metric':<14}{'BADGE (ab8b)':>14}{'conf-gated (ab8d)':>20}{'delta':>10}")
        for _m, _col in [('Recall', 'test_recall'), ('G-mean', 'test_gmean'),
                         ('F2', 'test_f2'), ('Specificity', 'test_specificity')]:
            _bv, _gv = float(_b[_col]), float(_g[_col])
            print(f"  {_m:<14}{_bv:>14.4f}{_gv:>20.4f}{_gv - _bv:>+10.4f}")

if _missing_pub:
    print("\n⚠️ Missing publication rows:")
    for _m in _missing_pub:
        print(f"  - {_m}")

_pub_path = _save_dir / f"ablation_publication_{_pub_suffix}.csv"
publication_df.to_csv(_pub_path, index=False)
print(f"\n💾 Saved publication table ({len(publication_df)} rows) → {_pub_path}")

print(f"\n✅ §I complete — summary_df and publication_df ready for §J/§K.")
print("=" * 70)


## §J — Visualization 1: Test Metrics (bar + scatter)

Grouped **bar chart** and **scatter plot** of the four held-out test metrics — **Recall** (`macro_recall`), **G-mean** (`macro_gmean`), **Specificity** (`macro_specificity`), **F2** (`macro_f2`) — across the selected ablations, in canonical run order (x-axis = ablation mode, y-axis = score). Reads `summary_df` from §I.


In [ ]:
# =============================================================================
# §J — Visualization 1: Test Metrics (bar + scatter, publication labels)
# =============================================================================
# Publication-focused 4 variants (paper Table 4):
#   Baseline -> (bagan,    ab1)   imbalanced, no augmentation
#   Variant1 -> (bagan,    ab8a)  aux-head label gate + margin
#   Variant2 -> (bagan_gp, ab8d)  confidence-gated diverse, GP backbone
#   Proposed -> (bagan_gp, ab8c)  Variant2's rule + co-training
# ab8b (BADGE) is run and stored in summary_df but is not plotted here.
# Displays: Recall, G-mean, F2, Specificity
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if 'summary_df' not in dir() or summary_df.empty:
    raise RuntimeError("§J — summary_df missing/empty. Run §I first.")

_METRICS = [
    ('test_recall',       'Recall'),
    ('test_gmean',        'G-mean'),
    ('test_f2',           'F2'),
    ('test_specificity',  'Specificity'),
]

_PUB_VARIANTS = [
    ('Baseline', 'bagan',    'ab1'),
    ('Variant1', 'bagan',    'ab8a'),
    ('Variant2', 'bagan_gp', 'ab8d'),
    ('Proposed', 'bagan_gp', 'ab8c'),
]

# Build publication-ordered rows from summary_df
_plot_rows = []
_missing_msgs = []
for _label, _backend, _mode in _PUB_VARIANTS:
    _sel = summary_df[(summary_df['backend'] == _backend) & (summary_df['mode'] == _mode)]
    _row = {'pub_label': _label, 'backend': _backend, 'mode': _mode}
    if _sel.empty:
        _missing_msgs.append(f"{_label} missing in summary_df -> filled as NaN")
        for _col, _ in _METRICS:
            _row[_col] = np.nan
    else:
        _r = _sel.iloc[0]
        for _col, _ in _METRICS:
            _row[_col] = float(_r.get(_col, np.nan))
    _plot_rows.append(_row)

_plot_df = pd.DataFrame(_plot_rows)

_xlabels = _plot_df['pub_label'].tolist()
_x = np.arange(len(_xlabels))
_colors = plt.get_cmap('plasma')(np.linspace(0.15, 0.85, len(_METRICS)))

fig, (ax_bar, ax_sc) = plt.subplots(1, 2, figsize=(14, 5))

# ── Auto y-limits: pad the observed range so every bar stays visible.
_all_vals = _plot_df[[c for c, _ in _METRICS]].values.astype(float)
_lo = float(np.nanmin(_all_vals)) if np.isfinite(_all_vals).any() else 0.0
_hi = float(np.nanmax(_all_vals)) if np.isfinite(_all_vals).any() else 1.0
_pad = max(0.01, (_hi - _lo) * 0.25)
_ylim = (max(0.0, _lo - _pad), min(1.0, _hi + _pad))

# ── Grouped bar chart ───────────────────────────────────────────────────────
_w = 0.8 / len(_METRICS)
for _i, (_col, _name) in enumerate(_METRICS):
    _vals = _plot_df[_col].values
    _off = (_i - (len(_METRICS) - 1) / 2) * _w
    ax_bar.bar(_x + _off, _vals, _w, label=_name, color=_colors[_i])
    for _xi, _v in zip(_x + _off, _vals):
        if not np.isnan(_v):
            ax_bar.text(_xi, _v, f'{_v:.3f}', ha='center', va='bottom', fontsize=8)
ax_bar.set_xticks(_x)
ax_bar.set_xticklabels(_xlabels, rotation=0, ha='center', fontsize=10)
ax_bar.set_ylabel('Score')
ax_bar.set_ylim(*_ylim)
ax_bar.set_title('Held-Out Test Metrics by Publication Variant (bar)')
ax_bar.legend(ncol=4, fontsize=9, loc='upper left')
ax_bar.grid(axis='y', alpha=0.3)

# ── Scatter plot ─────────────────────────────────────────────────────────────
for _i, (_col, _name) in enumerate(_METRICS):
    _vals = _plot_df[_col].values
    ax_sc.plot(_x, _vals, alpha=0.30, color=_colors[_i])
    ax_sc.scatter(_x, _vals, s=70, label=_name, color=_colors[_i], zorder=3)

    # value labels
    for _xi, _v in zip(_x, _vals):
        if not np.isnan(_v):
            ax_sc.text(_xi, _v, f'{_v:.3f}', ha='center', va='bottom', fontsize=9)
ax_sc.set_xticks(_x)
ax_sc.set_xticklabels(_xlabels, rotation=0, ha='center', fontsize=10)
ax_sc.set_ylabel('Score')
ax_sc.set_ylim(*_ylim)
ax_sc.set_title('Held-Out Test Metrics by Publication Variant (scatter)')
ax_sc.legend(ncol=4, fontsize=9, loc='lower right')
ax_sc.grid(alpha=0.3)

plt.tight_layout()
plt.show()

if _missing_msgs:
    print("⚠️ Missing variant rows:")
    for _m in _missing_msgs:
        print(f"  - {_m}")

print("✅ §J — publication-variant metrics bar + scatter rendered.")


## §J — Visualization 2: Per-Class FID (bar + scatter, with AE)

Grouped **bar chart** and **scatter plot** of per-class **FID** (lower = better) over the three generator stages plotted here: **Autoencoder** → **Generative model (BAGAN)** (Variant1's warm-up generator) → **Proposed** (the co-trained BAGAN-GP generator). Baselines have no generator and therefore no FID bars.

Variant2 now shares the BAGAN-GP backbone with Proposed (both use the gradient-penalised generator, per the updated Table 4 mapping), so it is not plotted as a separate group here. Its FID is still in `summary_df` (`backend='bagan_gp', mode='ab8d'`) — per the co-training G-guard finding (see 310_Ablation_five_seeds.ipynb), it is expected to match Proposed's FID exactly whenever the guard reverts to the warm-up generator.

> ⚠️ **Attribution guard.** The Generative model (BAGAN) → Proposed change conflates the **BAGAN → BAGAN-GP backbone swap** with **per-cycle generator co-training**. The cell prints the split; the paper must not claim the proposed method improves sample realism on the strength of this figure.


In [ ]:
# =============================================================================
# §J — Visualization 2: Per-Class FID (bar + scatter, publication labels)
# =============================================================================
# Publication groups (x-axis) — matches the 4-row Table 4:
#   Autoencoder               -> AE FID columns (ae_fid_c0/1/2; minority mean = mean(c1,c2))
#   Generative model (BAGAN)  -> (bagan, ab8a)  — Variant1's warm-up G
#   Proposed (BAGAN-GP)       -> (bagan_gp, ab8c)
#
# NOTE: Variant2 now shares the BAGAN-GP backbone with Proposed (Table 4 maps it
# to (bagan_gp, ab8d)), so it is not plotted as its own group -- only Variant1's
# BAGAN generator and Proposed's BAGAN-GP generator are shown, i.e. the backbone
# swap. Variant2's FID is still in summary_df for reference; per the co-training
# G-guard finding it is expected to equal Proposed's FID exactly.
#
# Uses a DIFFERENT colormap ('viridis') from §J Visualization 1 ('plasma')
# so the two figures are visually distinguishable at a glance.
#
# Displays (series): fid_c0, fid_c1, fid_c2, fid_minority_mean
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt

if 'summary_df' not in dir() or summary_df.empty:
    raise RuntimeError("§J(FID) — summary_df missing/empty. Run §I first.")

# 1) Fixed publication groups (x-axis)
_SOURCES = [
    ('Autoencoder',               'bagan',    None),
    ('Generative model (BAGAN)',  'bagan',    'ab8a'),
    ('Proposed (BAGAN-GP)',       'bagan_gp', 'ab8c'),
]

# 2) Fixed displays (series)
_DISPLAYS = [
    ('fid_c0',             'FID c0'),
    ('fid_c1',             'FID c1'),
    ('fid_c2',             'FID c2'),
    ('fid_minority_mean',  'FID minority mean'),
]

# 3) Build value matrix [n_sources, n_displays]
_values = []
_missing_msgs = []

for _label, _backend, _mode in _SOURCES:
    _row = summary_df[summary_df['backend'] == _backend]
    if _mode is not None:
        _row = _row[_row['mode'] == _mode]

    if _row.empty:
        _vals = [np.nan] * len(_DISPLAYS)
        _missing_msgs.append(f"{_label} missing in summary_df -> filled as NaN")
    else:
        _r = _row.iloc[0]
        if _mode is None:
            # Autoencoder source: use AE columns and compute minority mean from c1/c2.
            _ae_c0 = float(_r.get('ae_fid_c0', np.nan))
            _ae_c1 = float(_r.get('ae_fid_c1', np.nan))
            _ae_c2 = float(_r.get('ae_fid_c2', np.nan))
            _ae_min = float(np.nanmean([_ae_c1, _ae_c2]))
            _vals = [_ae_c0, _ae_c1, _ae_c2, _ae_min]
        else:
            _vals = [float(_r.get(_col, np.nan)) for _col, _ in _DISPLAYS]

    _values.append(_vals)

_M = np.array(_values, dtype=float)
_xlabels = [_s[0] for _s in _SOURCES]
_x = np.arange(len(_xlabels))
_colors = plt.get_cmap('viridis')(np.linspace(0.15, 0.85, len(_DISPLAYS)))

# 4) Plot: bar + scatter
fig, (ax_bar, ax_sc) = plt.subplots(1, 2, figsize=(15, 5))

# -- grouped bar
_w = 0.8 / len(_DISPLAYS)
for _i, (_col, _name) in enumerate(_DISPLAYS):
    _vals = _M[:, _i]
    _off = (_i - (len(_DISPLAYS) - 1) / 2) * _w
    _bx = _x + _off
    ax_bar.bar(_bx, _vals, _w, label=_name, color=_colors[_i])

    # value labels
    for _xi, _v in zip(_bx, _vals):
        if not np.isnan(_v):
            ax_bar.text(_xi, _v, f'{_v:.3f}', ha='center', va='bottom', fontsize=9)

ax_bar.set_xticks(_x)
ax_bar.set_xticklabels(_xlabels, rotation=0, ha='center', fontsize=9)
ax_bar.set_ylabel('FID (lower = better)')
ax_bar.set_title('Autoencoder -> Generative model (BAGAN) -> Proposed (BAGAN-GP) (bar)')
ax_bar.legend(fontsize=9)
ax_bar.grid(axis='y', alpha=0.3)

# -- scatter
for _i, (_col, _name) in enumerate(_DISPLAYS):
    _vals = _M[:, _i]
    ax_sc.plot(_x, _vals, alpha=0.30, color=_colors[_i])
    ax_sc.scatter(_x, _vals, s=70, label=_name, color=_colors[_i], zorder=3)

    # value labels
    for _xi, _v in zip(_x, _vals):
        if not np.isnan(_v):
            ax_sc.text(_xi, _v, f'{_v:.3f}', ha='center', va='bottom', fontsize=9)

ax_sc.set_xticks(_x)
ax_sc.set_xticklabels(_xlabels, rotation=0, ha='center', fontsize=9)
ax_sc.set_ylabel('FID (lower = better)')
ax_sc.set_title('Autoencoder -> Generative model (BAGAN) -> Proposed (BAGAN-GP) (scatter)')
ax_sc.legend(fontsize=9)
ax_sc.grid(alpha=0.3)

plt.tight_layout()
plt.show()

if _missing_msgs:
    print("⚠️ Missing source rows:")
    for _m in _missing_msgs:
        print(f"  - {_m}")

# ─── Attribution guard ──────────────────────────────────────────────────────
# The Generative model (BAGAN) -> Proposed FID change mixes TWO independent
# factors:
#   (a) the backbone swap  BAGAN -> BAGAN-GP  (warm-up generator, before any AL)
#   (b) per-cycle generator co-training       (the proposed component)
# Only (b) is attributable to the proposed method. Print the split so the
# paper never credits the whole delta to co-training.
_MIN_IDX = [_d[0] for _d in _DISPLAYS].index('fid_minority_mean')
_v12, _vprop = _M[1, _MIN_IDX], _M[2, _MIN_IDX]
print("\nMinority-FID decomposition (do NOT attribute the whole delta to co-training):")
print(f"  Generative model (BAGAN warm-up G)    : {_v12:.3f}")
print(f"  Proposed     (BAGAN-GP + co-training) : {_vprop:.3f}   net {_vprop - _v12:+.3f}")
print("  ⚠️  The net change is dominated by the BAGAN -> BAGAN-GP backbone swap.")
print("     To isolate co-training, compare the Proposed FID against the BAGAN-GP")
print("     warm-up generator (pre-AL), not against the vanilla-BAGAN column.")

print("\n✅ §J — Autoencoder/Generative model (BAGAN)/Proposed (BAGAN-GP) FID bar + scatter rendered.")


## §K — Visualization 3: Generated-Sample Comparison Grid

A 4-row × 9-column figure comparing 3 images per class (class 0 / 1 / 2) for:

1. **Real** images (from the retrain pool)
2. **Autoencoder**-decoded samples (vanilla `bagan` pass AE)
3. **Generative model (BAGAN)** — Variant1's warm-up generator (`bagan` pass)
4. **Proposed** — the co-trained BAGAN-GP generator (`bagan_gp` pass)

Reads the per-backend dumps saved in §D.7 (`saved/samples_<suffix>_bagan.npz`, `saved/samples_<suffix>_bagan_gp.npz`, suffix = `config.date_suffix`). **Run after both passes** for the full grid; if a pass is missing, its row is skipped with a note.

> The BAGAN row shows Variant1's generator only — Variant2 now shares Proposed's BAGAN-GP backbone (see §J Viz 2), so it would be redundant as a fifth row.


In [ ]:
from datetime import datetime
print(f"\n End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

import time   # local alias to avoid shadowing if already imported

time_stamp_3 = time.time()

In [ ]:
print(f"CV    time: {time_stamp_2 - time_stamp_1:.2f} seconds, minutes: {(time_stamp_2 - time_stamp_1)/60:.2f} minutes, hours: {(time_stamp_2 - time_stamp_1)/3600:.2f} hours")
print(f"Test  time: {time_stamp_3 - time_stamp_2:.2f} seconds, minutes: {(time_stamp_3 - time_stamp_2)/60:.2f} minutes, hours: {(time_stamp_3 - time_stamp_2)/3600:.2f} hours")
    
total_elapsed = time_stamp_3 - time_stamp_1
print(f"Total time: {total_elapsed:.2f} seconds, minutes: {total_elapsed/60:.2f} minutes, hours: {total_elapsed/3600:.2f} hours")